
# CELDA 1 — Configuración general del HBM v2 para no2.5

# **Objetivo:**
dejar definidas las rutas, parámetros y nombres de columnas que usará
el HBM v2 con observaciones reales de estaciones.

# **Qué hace esta celda:**
 1. Define la carpeta de trabajo y las rutas de entrada.
 2. Declara el contaminante objetivo (`NO2`).
 3. Define las covariables base del HBM:
   - `Vel_viento_idw`
   - `diff_NO2_grid`
   - `grad_NO2_grid`
    - `adv_proxy_NO2_grid`
 4. Crea la carpeta de salida donde se guardarán paneles, folds,
    resultados de validación y superficies finales.

 **Nota metodológica importante:**
 En esta versión no se usa `NO2_idw` como predictor directo del HBM.
 La respuesta será la observación real de estación (`NO2_obs`).


In [1]:
# %%
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# 0) Rutas
# =========================
PROJECT_ROOT = Path.cwd()

# Ajusta esta línea si tus archivos están en otra carpeta
DATA_DIR = PROJECT_ROOT

OBS_CSV  = DATA_DIR / "panel_ambiental_mensual_2020_2024_coords_corregidas.csv"
GRID_CSV = DATA_DIR / "grid_3km_AD_mensual_2020_2024.csv"
W_CSV    = DATA_DIR / "W_grid_3km_queen.csv"

OUT_DIR = PROJECT_ROOT / "HBM_O3_V2_OUT"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 1) Configuración HBM v2
# =========================
POLL = "O3"  
OBS_COL = "O3"
VALID_COL = "valido_O3"

X_COLS_RAW = [
    "Vel_viento_idw",
    "diff_O3_grid",
    "grad_O3_grid",
    "adv_proxy_O3_grid",
]

USE_LOG = True
EPS = 1e-6

# folds temporales
FOLDS = {
    "fold_1": {"train_years": [2020, 2021], "test_years": [2022]},
    "fold_2": {"train_years": [2020, 2021, 2022], "test_years": [2023]},
    "fold_3": {"train_years": [2020, 2021, 2022, 2023], "test_years": [2024]},
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("OUT_DIR      :", OUT_DIR)
print("\nArchivos de entrada:")
for p in [OBS_CSV, GRID_CSV, W_CSV]:
    print(" -", p.name, "| exists:", p.exists())

for p in [OBS_CSV, GRID_CSV, W_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"No encuentro el archivo: {p}")

print("\nConfiguración lista.")
print("POLL         :", POLL)
print("OBS_COL      :", OBS_COL)
print("VALID_COL    :", VALID_COL)
print("X_COLS_RAW   :", X_COLS_RAW)
print("USE_LOG      :", USE_LOG)


PROJECT_ROOT: d:\TRABAJO DE GRADO BEN-MAP\CODIGO
DATA_DIR     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO
OUT_DIR      : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT

Archivos de entrada:
 - panel_ambiental_mensual_2020_2024_coords_corregidas.csv | exists: True
 - grid_3km_AD_mensual_2020_2024.csv | exists: True
 - W_grid_3km_queen.csv | exists: True

Configuración lista.
POLL         : O3
OBS_COL      : O3
VALID_COL    : valido_O3
X_COLS_RAW   : ['Vel_viento_idw', 'diff_O3_grid', 'grad_O3_grid', 'adv_proxy_O3_grid']
USE_LOG      : True



# CELDA 2 — Cargar, limpiar y construir el panel base de modelación

# **Objetivo:**
# construir dos paneles:

 1. `grid_base`:
    la superficie completa celda–mes de la malla 3 km con covariables A–D.

 2. `obs_panel`:
    las observaciones reales de estación para no2.5, ya enlazadas a:
   - `cell_id`
    - `fecha`
    - covariables de la malla

 **Qué hace esta celda:**
 1. Carga el panel de estaciones corregido.
 2. Carga la malla 3 km con términos A–D.
 3. Convierte fechas a formato mensual.
 4. Filtra observaciones válidas de no2.5.
 5. Une cada observación con la covariable de su celda y mes.
 6. Crea índices globales:
    - `cell_idx`
    - `time_idx`
 7. Guarda paneles limpios para el HBM.


In [2]:
# %%
# =========================
# 2) Cargar archivos
# =========================
obs = pd.read_csv(OBS_CSV)
grid = pd.read_csv(GRID_CSV)
w = pd.read_csv(W_CSV)

# =========================
# 3) Fechas
# =========================
obs["fecha"] = pd.to_datetime(
    dict(year=obs["Año"].astype(int), month=obs["Mes"].astype(int), day=1),
    errors="coerce"
)
grid["fecha"] = pd.to_datetime(grid["fecha"], errors="coerce")

if obs["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel de estaciones.")
if grid["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en la malla 3km.")

# =========================
# 4) Validación básica de columnas
# =========================
obs_needed = ["Estacion", "cell_id", "fecha", OBS_COL, VALID_COL]
grid_needed = ["cell_id", "fecha"] + X_COLS_RAW

for c in obs_needed:
    if c not in obs.columns:
        raise ValueError(f"Falta la columna '{c}' en el panel de estaciones.")

for c in grid_needed:
    if c not in grid.columns:
        raise ValueError(f"Falta la columna '{c}' en la malla 3km.")

# =========================
# 5) Filtrar observaciones válidas O3.5
# =========================
obs[VALID_COL] = obs[VALID_COL].astype(bool)

obs_no = obs.loc[
    (obs[VALID_COL] == True) &
    (obs[OBS_COL].notna()) &
    (obs["cell_id"].notna())
].copy()

obs_no = obs_no.rename(columns={OBS_COL: "O3_obs"})

# =========================
# 6) Construir grid_base
# =========================
grid_base = grid[["cell_id", "fecha"] + X_COLS_RAW].copy()
grid_base["year"] = grid_base["fecha"].dt.year
grid_base["month"] = grid_base["fecha"].dt.month

# =========================
# 7) Unir observaciones con covariables de malla
# =========================
obs_panel = obs_no.merge(
    grid_base,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_merge = obs_panel[X_COLS_RAW].isna().any(axis=1).sum()
if missing_merge > 0:
    raise ValueError(
        f"Hay {missing_merge} observaciones sin covariables de la malla. "
        "Revisa cell_id y fecha."
    )

obs_panel["year"] = obs_panel["fecha"].dt.year
obs_panel["month"] = obs_panel["fecha"].dt.month

# =========================
# 8) Índices globales de celda y tiempo
# =========================
cell_ids = np.sort(grid_base["cell_id"].unique())
time_ids = np.sort(grid_base["fecha"].unique())

cell_map = {cid: i for i, cid in enumerate(cell_ids)}
time_map = {tt: j for j, tt in enumerate(time_ids)}

grid_base["cell_idx"] = grid_base["cell_id"].map(cell_map).astype(int)
grid_base["time_idx"] = grid_base["fecha"].map(time_map).astype(int)

obs_panel["cell_idx"] = obs_panel["cell_id"].map(cell_map).astype(int)
obs_panel["time_idx"] = obs_panel["fecha"].map(time_map).astype(int)

# =========================
# 9) Guardar paneles base
# =========================
grid_base_path = OUT_DIR / "HBM_O3_grid_base_v2.csv"
obs_panel_path = OUT_DIR / "HBM_O3_obs_panel_v2.csv"

grid_base.to_csv(grid_base_path, index=False)
obs_panel.to_csv(obs_panel_path, index=False)

# =========================
# 10) Resumen
# =========================
print("Resumen del panel base HBM v2")
print("- obs_panel filas          :", len(obs_panel))
print("- estaciones únicas       :", obs_panel["Estacion"].nunique())
print("- celdas con observación   :", obs_panel["cell_id"].nunique())
print("- grid_base filas          :", len(grid_base))
print("- celdas totales en malla  :", len(cell_ids))
print("- meses totales            :", len(time_ids))
print("\nArchivos guardados:")
print(" -", grid_base_path)
print(" -", obs_panel_path)


Resumen del panel base HBM v2
- obs_panel filas          : 641
- estaciones únicas       : 13
- celdas con observación   : 13
- grid_base filas          : 15240
- celdas totales en malla  : 254
- meses totales            : 60

Archivos guardados:
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_base_v2.csv
 - d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_panel_v2.csv



# # CELDA 3 — Preparar vecindad espacial y folds temporales

# **Objetivo:**
# dejar lista la estructura espacial del HBM y la validación temporal.

# **Qué hace esta celda:**
 1. Toma `W_grid_3km_queen.csv`.
 2. Verifica que las celdas de la vecindad existan en la malla.
 3. Convierte `cell_id` y `neighbor_id` a índices internos `i`, `j`.
 4. Elimina duplicados dirigidos para dejar pares únicos no dirigidos.
 5. Guarda la vecindad final para el modelo.
 6. Guarda también la definición de folds temporales.

# **Salidas:**
# - `HBM_NO2_W_edges_queen_v2.csv`
# - `HBM_NO2_folds_v2.json`

In [3]:
# %%
# =========================
# CELDA 3) Vecindad espacial y folds temporales
# =========================

# Validación mínima de columnas en W
if not {"cell_id", "neighbor_id"}.issubset(w.columns):
    raise ValueError("W_grid_3km_queen.csv debe tener las columnas: 'cell_id' y 'neighbor_id'.")

# Filtrar solo relaciones cuyos nodos existan en la malla
w_use = w[
    w["cell_id"].isin(cell_ids) &
    w["neighbor_id"].isin(cell_ids)
].copy()

if len(w_use) == 0:
    raise ValueError("La vecindad quedó vacía después de filtrar por celdas de la malla.")

# Mapear a índices internos
w_use["i"] = w_use["cell_id"].map(cell_map)
w_use["j"] = w_use["neighbor_id"].map(cell_map)

if w_use["i"].isna().any() or w_use["j"].isna().any():
    raise ValueError("Hay relaciones de vecindad que no pudieron mapearse a índices internos.")

w_use["i"] = w_use["i"].astype(int)
w_use["j"] = w_use["j"].astype(int)

# Quitar lazos propios si existieran
w_use = w_use.loc[w_use["i"] != w_use["j"]].copy()

# Dejar pares únicos no dirigidos
ii = np.minimum(w_use["i"].values, w_use["j"].values)
jj = np.maximum(w_use["i"].values, w_use["j"].values)

pairs = np.unique(np.column_stack([ii, jj]), axis=0)
w_edges = pd.DataFrame(pairs, columns=["i", "j"])

# Guardar W final
w_edges_path = OUT_DIR / "HBM_O3_W_edges_queen_v2.csv"
w_edges.to_csv(w_edges_path, index=False)

# Guardar folds
folds_path = OUT_DIR / "HBM_O3_folds_v2.json"
with open(folds_path, "w", encoding="utf-8") as f:
    json.dump(FOLDS, f, ensure_ascii=False, indent=2)

# Resumen
print("Resumen CELDA 3")
print("- relaciones originales en W      :", len(w))
print("- relaciones válidas tras filtro  :", len(w_use))
print("- edges únicos no dirigidos       :", len(w_edges))
print("- nodos únicos en W               :", len(set(w_use['i']).union(set(w_use['j']))))

print("\nArchivos guardados:")
print("-", w_edges_path)
print("-", folds_path)

print("\nFolds temporales:")
print(json.dumps(FOLDS, ensure_ascii=False, indent=2))

Resumen CELDA 3
- relaciones originales en W      : 1610
- relaciones válidas tras filtro  : 1610
- edges únicos no dirigidos       : 805
- nodos únicos en W               : 254

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_W_edges_queen_v2.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_folds_v2.json

Folds temporales:
{
  "fold_1": {
    "train_years": [
      2020,
      2021
    ],
    "test_years": [
      2022
    ]
  },
  "fold_2": {
    "train_years": [
      2020,
      2021,
      2022
    ],
    "test_years": [
      2023
    ]
  },
  "fold_3": {
    "train_years": [
      2020,
      2021,
      2022,
      2023
    ],
    "test_years": [
      2024
    ]
  }
}



# # CELDA 4 — Funciones auxiliares para escalamiento, métricas e intervalos

# **Objetivo:**
definir funciones reutilizables para preparar los datos del HBM v2
y evaluar sus predicciones de manera consistente.

**Qué hace esta celda:**
 1. Define una función para estandarizar covariables usando solo los años de entrenamiento.
 2. Define una función para llevar esas covariables escaladas al panel observado.
 3. Define una función para calcular métricas puntuales:
    - MAE
    - RMSE
    - sesgo
    - correlación
    - R²
 4. Define métricas probabilísticas:
    - cobertura del intervalo 90%
    - ancho promedio del intervalo
    - WIS
 5. Define una función para extraer `p05`, `p50` y `p95`
    a partir del posterior del modelo.

 **Importancia metodológica:**
 el escalamiento se hace usando únicamente el período de entrenamiento,
 para evitar fuga de información hacia los folds de prueba.

In [4]:
# %%
# =========================
# CELDA 4) Funciones auxiliares
# =========================

def scale_grid_by_train_years(grid_df, x_cols, train_years):
    """
    Estandariza covariables usando solo las filas de grid_df
    pertenecientes a los años de entrenamiento.

    Parámetros
    ----------
    grid_df : pd.DataFrame
        Panel celda-mes de la malla.
    x_cols : list[str]
        Lista de covariables crudas a escalar.
    train_years : list[int]
        Años que pertenecen al conjunto de entrenamiento.

    Retorna
    -------
    grid_scaled : pd.DataFrame
        DataFrame con nuevas columnas *_z.
    params : dict
        Media y desviación estándar usadas para cada covariable.
    """
    grid_scaled = grid_df.copy()
    params = {}

    train_mask = grid_scaled["year"].isin(train_years)

    for col in x_cols:
        mu = grid_scaled.loc[train_mask, col].mean()
        sd = grid_scaled.loc[train_mask, col].std(ddof=0)

        if pd.isna(sd) or sd == 0:
            sd = 1.0

        z_col = f"{col}_z"
        grid_scaled[z_col] = (grid_scaled[col] - mu) / sd
        params[col] = {"mu": float(mu), "sd": float(sd)}

    return grid_scaled, params


def merge_scaled_covariates_to_obs(obs_df, grid_scaled, x_cols):
    """
    Lleva las covariables estandarizadas desde la malla al panel observado,
    usando la llave (cell_id, fecha).
    """
    z_cols = [f"{c}_z" for c in x_cols]

    out = obs_df.drop(columns=x_cols, errors="ignore").merge(
        grid_scaled[["cell_id", "fecha"] + z_cols],
        on=["cell_id", "fecha"],
        how="left",
        validate="many_to_one"
    )

    missing = out[z_cols].isna().any(axis=1).sum()
    if missing > 0:
        raise ValueError(f"Hay {missing} observaciones sin covariables escaladas.")

    return out


def interval_metrics(y_true, p05, p50, p95, alpha=0.10):
    """
    Calcula métricas puntuales y probabilísticas
    sobre un intervalo central del 90%.
    """
    y_true = np.asarray(y_true, dtype=float)
    p05 = np.asarray(p05, dtype=float)
    p50 = np.asarray(p50, dtype=float)
    p95 = np.asarray(p95, dtype=float)

    m = ~np.isnan(y_true) & ~np.isnan(p05) & ~np.isnan(p50) & ~np.isnan(p95)
    y_true = y_true[m]
    p05 = p05[m]
    p50 = p50[m]
    p95 = p95[m]

    if len(y_true) == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "r": np.nan,
            "r2": np.nan,
            "coverage_90": np.nan,
            "width_90_mean": np.nan,
            "wis_90": np.nan,
        }

    err = p50 - y_true
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    bias = np.mean(err)

    if np.std(y_true) == 0 or np.std(p50) == 0:
        r = np.nan
        r2 = np.nan
    else:
        r = np.corrcoef(y_true, p50)[0, 1]
        r2 = r**2

    coverage = np.mean((y_true >= p05) & (y_true <= p95))
    width = np.mean(p95 - p05)

    wis = np.mean(
        (p95 - p05)
        + (2 / alpha) * (p05 - y_true) * (y_true < p05)
        + (2 / alpha) * (y_true - p95) * (y_true > p95)
    )

    return {
        "n": int(len(y_true)),
        "mae": float(mae),
        "rmse": float(rmse),
        "bias": float(bias),
        "r": float(r) if not np.isnan(r) else np.nan,
        "r2": float(r2) if not np.isnan(r2) else np.nan,
        "coverage_90": float(coverage),
        "width_90_mean": float(width),
        "wis_90": float(wis),
    }


def posterior_predict_concentration(idata, X_mat, cell_idx_arr, time_idx_arr, eps=1e-6):
    """
    A partir del posterior del HBM, obtiene p05, p50 y p95
    en escala original de concentración.
    """
    alpha_s = idata.posterior["alpha"].values.reshape(-1)
    beta_s = idata.posterior["beta"].values.reshape(-1, idata.posterior["beta"].values.shape[-1])
    phi_s = idata.posterior["phi"].values.reshape(-1, idata.posterior["phi"].values.shape[-1])
    delta_s = idata.posterior["delta"].values.reshape(-1, idata.posterior["delta"].values.shape[-1])

    mu_s = (
        alpha_s[:, None]
        + (beta_s @ X_mat.T)
        + phi_s[:, cell_idx_arr]
        + delta_s[:, time_idx_arr]
    )

    c_s = np.exp(mu_s) - eps

    p05 = np.quantile(c_s, 0.05, axis=0)
    p50 = np.quantile(c_s, 0.50, axis=0)
    p95 = np.quantile(c_s, 0.95, axis=0)

    return p05, p50, p95


print("CELDA 4 cargada correctamente.")
print("Funciones disponibles:")
print("- scale_grid_by_train_years")
print("- merge_scaled_covariates_to_obs")
print("- interval_metrics")
print("- posterior_predict_concentration")

CELDA 4 cargada correctamente.
Funciones disponibles:
- scale_grid_by_train_years
- merge_scaled_covariates_to_obs
- interval_metrics
- posterior_predict_concentration



 # CELDA 5 — Definición del modelo base M1: ICAR + RW1

 **Objetivo:**
 dejar definida la función que ajusta el HBM base para un fold temporal.

 **Especificación del modelo M1:**

 - Respuesta observada:
   `NO2_obs`
 - Escala:
   logarítmica
 - Efectos fijos:
   covariables A–D + viento
 - Efecto espacial:
   `ICAR`
 - Efecto temporal:
   `RW1`

 **Forma general del modelo:**

 `log(NO2_obs) = alpha + X beta + phi_i + delta_t + error`

 donde:
 - `phi_i` representa el efecto espacial estructurado sobre la malla
 - `delta_t` representa la evolución temporal mensual

 **Qué hace esta celda:**
 1. Importa PyMC, PyTensor y ArviZ.
 2. Define la función `fit_hbm_m1_fold`.
 3. La función:
    - recibe train/test por años,
    - ajusta el modelo bayesiano,
    - predice sobre el período test,
    - calcula percentiles posteriores,
    - devuelve métricas e intervalos.

 **Nota:**
 si PyMC no está instalado en tu entorno, esta celda te lo indicará.

In [5]:
# %%
# =========================
# CELDA 5) Modelo base M1: ICAR + RW1
# =========================

try:
    import pymc as no
    import pytensor.tensor as pt
    import arviz as az
except Exception as e:
    raise ImportError(
        "No se pudieron importar pymc / pytensor / arviz.\n"
        "Instala en tu entorno:\n"
        "pip install pymc arviz pytensor\n\n"
        f"Detalle original: {e}"
    )


def fit_hbm_m1_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
):
    """
    Ajusta el HBM base M1 para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada O3_obs
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # Matrices y vectores del conjunto de entrenamiento
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["O3_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # Matrices y vectores del conjunto de prueba
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["O3_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # Edges espaciales
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]

    with no.Model() as model:
        # -------------------------
        # Efectos fijos
        # -------------------------
        alpha = no.Normal("alpha", mu=0.0, sigma=5.0)
        beta = no.Normal("beta", mu=0.0, sigma=1.0, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = no.HalfNormal("sigma_y", sigma=1.0)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = no.Exponential("tau_phi", 1.0)
        phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        no.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1
        # -------------------------
        sigma_t = no.HalfNormal("sigma_t", sigma=1.0)
        delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Verosimilitud
        # -------------------------
        no.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo MCMC
        # -------------------------
        idata = no.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción sobre test
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "O3_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    # -------------------------
    # Métricas del fold
    # -------------------------
    met = interval_metrics(
        y_true=test_pred["O3_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 5 cargada correctamente.")
print("Función disponible: fit_hbm_m1_fold")

CELDA 5 cargada correctamente.
Función disponible: fit_hbm_m1_fold



 # CELDA 6 — Validación temporal del modelo base M1

 **Objetivo:**
 ejecutar la validación temporal del HBM base usando los 3 folds definidos.

 **Qué hace esta celda:**
 1. Recorre cada fold temporal.
 2. Escala las covariables usando únicamente los años de entrenamiento.
 3. Lleva las covariables escaladas al panel observado.
 4. Ajusta el modelo HBM base M1:
    - espacial ICAR
    - temporal RW1
 5. Predice sobre el período de prueba.
 6. Guarda:
    - predicciones del fold
    - resumen del posterior
    - parámetros de escalamiento
 7. Consolida:
    - métricas de todos los folds
    - predicciones conjuntas de prueba

 **Salidas principales:**
 - `HBM_NO2_M1_fold_1_pred_test.csv`
 - `HBM_NO2_M1_fold_2_pred_test.csv`
 - `HBM_NO2_M1_fold_3_pred_test.csv`
 - `HBM_NO2_M1_metrics_folds.csv`
 - `HBM_NO2_M1_pred_test_all_folds.csv`

 **Nota práctica:**
 esta es la primera corrida real del HBM. Puede tardar bastante
 dependiendo del equipo y del entorno de Python.

In [6]:
# %%
# =========================
# CELDA 6) Validación temporal M1
# =========================

all_metrics = []
all_test_preds = []

# puedes subir estos valores más adelante si quieres una corrida más exigente
DRAWS = 800
TUNE = 800
CHAINS = 4
TARGET_ACCEPT = 0.95
RANDOM_SEED = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo años train
    # -------------------------------------------------
    grid_scaled, scale_params = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold, test_pred_fold, met_fold = fit_hbm_m1_fold(
        obs_scaled=obs_scaled,
        grid_scaled=grid_scaled,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS,
        tune=TUNE,
        chains=CHAINS,
        target_accept=TARGET_ACCEPT,
        random_seed=RANDOM_SEED,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold["fold"] = fold_name
    pred_fold_path = OUT_DIR / f"HBM_O3_M1_{fold_name}_pred_test.csv"
    test_pred_fold.to_csv(pred_fold_path, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold = az.summary(
        idata_fold,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path = OUT_DIR / f"HBM_O3_M1_{fold_name}_summary.csv"
    summary_fold.to_csv(summary_fold_path)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path = OUT_DIR / f"HBM_O3_M1_{fold_name}_scale_params.json"
    with open(scale_fold_path, "w", encoding="utf-8") as f:
        json.dump(scale_params, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold["fold"] = fold_name
    met_fold["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold["test_years"] = ",".join(map(str, fold_info["test_years"]))
    all_metrics.append(met_fold)
    all_test_preds.append(test_pred_fold)

    print("\nGuardado del fold:")
    print("-", pred_fold_path)
    print("-", summary_fold_path)
    print("-", scale_fold_path)

    print("\nMétricas del fold:")
    for k, v in met_fold.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar todo
# -------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
preds_df = pd.concat(all_test_preds, ignore_index=True)

metrics_path = OUT_DIR / "HBM_O3_M1_metrics_folds.csv"
preds_path = OUT_DIR / "HBM_O3_M1_pred_test_all_folds.csv"

metrics_df.to_csv(metrics_path, index=False)
preds_df.to_csv(preds_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL COMPLETADA")
print("- métricas consolidadas :", metrics_path)
print("- predicciones consolidadas :", preds_path)

print("\nResumen final de métricas:")
display(metrics_df)


Corriendo fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 113 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_1_scale_params.json

Métricas del fold:
- n: 130
- mae: 2.4138529377311895
- rmse: 2.9495662971142664
- bias: 0.6681663530757297
- r: 0.7879309162203172
- r2: 0.6208351287357885
- coverage_90: 0.9615384615384616
- width_90_mean: 21.487810719733787
- wis_90: 22.81843301053586
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 

Output()

Sampling 4 chains for 800 tune and 800 draw iterations (3_200 + 3_200 draws total) took 85 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_fold_3_scale_params.json

Métricas del fold:
- n: 136
- mae: 3.8329674553003477
- rmse: 4.525072388377172
- bias: -1.8527071636326131
- r: 0.6419331907844171
- r2: 0.41207822143066286
- coverage_90: 0.9044117647058824
- width_90_mean: 19.99818002216664
- wis_90: 21.551398981444436
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1_pred_test_all_folds.csv

Resumen final de métricas:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,fold,train_years,test_years
0,130,2.413853,2.949566,0.668166,0.787931,0.620835,0.961538,21.487811,22.818433,fold_1,"2020,2021",2022
1,118,2.517811,3.080924,0.178419,0.753846,0.568284,0.983051,21.137129,21.294531,fold_2,"2020,2021,2022",2023
2,136,3.832967,4.525072,-1.852707,0.641933,0.412078,0.904412,19.998180,21.551399,fold_3,"2020,2021,2022,2023",2024



 # CELDA 6B — Redefinir el modelo base en versión más estable (M1b)

 **Objetivo:**
 crear una versión más robusta del HBM base para mejorar la convergencia
 sin cambiar la estructura sustantiva del modelo.

 **Qué cambia respecto al M1 inicial:**
 1. Se mantienen:
    - respuesta observada `NO2_obs`
    - efecto espacial ICAR
    - efecto temporal RW1
    - covariables A–D + viento
 2. Se ajustan priors para hacer el modelo más regularizado.
 3. Se mejora el muestreo con:
    - `target_accept` más alto
    - `max_treedepth` más alto
    - `init="adapt_diag"`

 **Objetivo metodológico:**
 reducir problemas de:
 - Rhat alto
 - ESS bajo
 - tree depth máximo

 **Importante:**
 esta celda solo redefine la función.
 En la siguiente la volvemos a correr por folds.

In [7]:
# %%
# =========================
# CELDA 6B) Versión estable del modelo base
# =========================

def fit_hbm_m1b_fold(
    obs_scaled,
    grid_scaled,
    w_edges_df,
    x_cols_raw,
    train_years,
    test_years,
    n_cells,
    n_times,
    draws=1000,
    tune=1500,
    chains=4,
    target_accept=0.99,
    max_treedepth=15,
    random_seed=42,
):
    """
    Ajusta el HBM base M1b para un fold temporal:
    - espacial ICAR
    - temporal RW1
    - respuesta observada O3_obs
    - priors más regularizantes
    - sampler más conservador
    """

    z_cols = [f"{c}_z" for c in x_cols_raw]

    train_mask_obs = obs_scaled["year"].isin(train_years)
    test_mask_obs = obs_scaled["year"].isin(test_years)

    train_obs = obs_scaled.loc[train_mask_obs].copy()
    test_obs = obs_scaled.loc[test_mask_obs].copy()

    if len(train_obs) == 0:
        raise ValueError("No hay observaciones de entrenamiento para este fold.")
    if len(test_obs) == 0:
        raise ValueError("No hay observaciones de prueba para este fold.")

    # -------------------------
    # Datos train
    # -------------------------
    X_train = train_obs[z_cols].to_numpy(dtype=float)
    y_train_raw = train_obs["O3_obs"].to_numpy(dtype=float)
    y_train = np.log(y_train_raw + EPS) if USE_LOG else y_train_raw

    cell_train = train_obs["cell_idx"].to_numpy(dtype=int)
    time_train = train_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Datos test
    # -------------------------
    X_test = test_obs[z_cols].to_numpy(dtype=float)
    y_test = test_obs["O3_obs"].to_numpy(dtype=float)
    cell_test = test_obs["cell_idx"].to_numpy(dtype=int)
    time_test = test_obs["time_idx"].to_numpy(dtype=int)

    # -------------------------
    # Vecindad espacial
    # -------------------------
    ei = w_edges_df["i"].to_numpy(dtype=int)
    ej = w_edges_df["j"].to_numpy(dtype=int)

    p = X_train.shape[1]
    alpha_mu = float(np.mean(y_train))

    with no.Model() as model:
        # -------------------------
        # Efectos fijos más regularizados
        # -------------------------
        alpha = no.Normal("alpha", mu=alpha_mu, sigma=1.0)
        beta = no.Normal("beta", mu=0.0, sigma=0.5, shape=p)

        # -------------------------
        # Error observacional
        # -------------------------
        sigma_y = no.HalfNormal("sigma_y", sigma=0.75)

        # -------------------------
        # Efecto espacial ICAR
        # -------------------------
        tau_phi = no.Exponential("tau_phi", 2.0)
        phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=n_cells)
        phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))

        no.Potential(
            "icar_penalty",
            -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2)
        )

        # -------------------------
        # Efecto temporal RW1 más controlado
        # -------------------------
        sigma_t = no.HalfNormal("sigma_t", sigma=0.25)
        delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=n_times)
        delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

        # -------------------------
        # Media del modelo
        # -------------------------
        mu_train = alpha + pt.dot(X_train, beta) + phi[cell_train] + delta[time_train]

        # -------------------------
        # Likelihood
        # -------------------------
        no.Normal("y_obs", mu=mu_train, sigma=sigma_y, observed=y_train)

        # -------------------------
        # Muestreo
        # -------------------------
        idata = no.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            init="adapt_diag",
            target_accept=target_accept,
            max_treedepth=max_treedepth,
            random_seed=random_seed,
            return_inferencedata=True,
            progressbar=True,
        )

    # -------------------------
    # Predicción test en escala original
    # -------------------------
    p05_test, p50_test, p95_test = posterior_predict_concentration(
        idata=idata,
        X_mat=X_test,
        cell_idx_arr=cell_test,
        time_idx_arr=time_test,
        eps=EPS
    )

    test_pred = test_obs[
        ["Estacion", "fecha", "year", "month", "cell_id", "cell_idx", "time_idx", "O3_obs"]
    ].copy()

    test_pred["p05_hbm"] = p05_test
    test_pred["p50_hbm"] = p50_test
    test_pred["p95_hbm"] = p95_test
    test_pred["width_90"] = test_pred["p95_hbm"] - test_pred["p05_hbm"]

    met = interval_metrics(
        y_true=test_pred["O3_obs"].values,
        p05=test_pred["p05_hbm"].values,
        p50=test_pred["p50_hbm"].values,
        p95=test_pred["p95_hbm"].values,
        alpha=0.10
    )

    return idata, test_pred, met


print("CELDA 6B cargada correctamente.")
print("Función disponible: fit_hbm_m1b_fold")

CELDA 6B cargada correctamente.
Función disponible: fit_hbm_m1b_fold


# CELDA 6C — Validación temporal del modelo estable M1b

 **Objetivo:**
 volver a ejecutar la validación temporal del HBM usando la versión
 más estable del modelo base:

 - espacial ICAR
 - temporal RW1
 - priors más regularizantes
 - muestreo más conservador

 **Qué hace esta celda:**
 1. Recorre los 3 folds temporales.
 2. Escala covariables usando solo train.
 3. Ajusta `fit_hbm_m1b_fold`.
 4. Guarda:
    - predicciones por fold
   - resumen posterior por fold
    - parámetros de escalamiento
 5. Consolida:
    - métricas de todos los folds
    - predicciones de prueba conjuntas
 6. Si existe el archivo de métricas del modelo M1 anterior,
    genera una tabla comparativa M1 vs M1b.

 **Salidas principales:**
 - `HBM_NO2_M1b_fold_1_pred_test.csv`
 - `HBM_NO2_M1b_fold_2_pred_test.csv`
 - `HBM_NO2_M1b_fold_3_pred_test.csv`
 - `HBM_NO2_M1b_metrics_folds.csv`
 - `HBM_NO2_M1b_pred_test_all_folds.csv`
 - `HBM_NO2_compare_M1_vs_M1b.csv` (si existe M1)

 **Qué esperamos observar:**
 - menor problema de convergencia
 - Rhat más cercano a 1
 - ESS más alto
 - calibración de intervalos más estable

In [8]:
# %%
# =========================
# CELDA 6C) Validación temporal M1b
# =========================

all_metrics_m1b = []
all_test_preds_m1b = []

# configuración más conservadora
DRAWS_B = 1000
TUNE_B = 1500
CHAINS_B = 4
TARGET_ACCEPT_B = 0.99
MAX_TREEDEPTH_B = 15
RANDOM_SEED_B = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalamiento usando solo train
    # -------------------------------------------------
    grid_scaled_b, scale_params_b = scale_grid_by_train_years(
        grid_df=grid_base,
        x_cols=X_COLS_RAW,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado
    # -------------------------------------------------
    obs_scaled_b = merge_scaled_covariates_to_obs(
        obs_df=obs_panel,
        grid_scaled=grid_scaled_b,
        x_cols=X_COLS_RAW
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_b, test_pred_fold_b, met_fold_b = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_b,
        grid_scaled=grid_scaled_b,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_RAW,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_B,
        tune=TUNE_B,
        chains=CHAINS_B,
        target_accept=TARGET_ACCEPT_B,
        max_treedepth=MAX_TREEDEPTH_B,
        random_seed=RANDOM_SEED_B,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_b["fold"] = fold_name
    pred_fold_path_b = OUT_DIR / f"HBM_O3_M1b_{fold_name}_pred_test.csv"
    test_pred_fold_b.to_csv(pred_fold_path_b, index=False)

    # -------------------------------------------------
    # 5) Guardar resumen del posterior
    # -------------------------------------------------
    summary_fold_b = az.summary(
        idata_fold_b,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_b = OUT_DIR / f"HBM_O3_M1b_{fold_name}_summary.csv"
    summary_fold_b.to_csv(summary_fold_path_b)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_b = OUT_DIR / f"HBM_O3_M1b_{fold_name}_scale_params.json"
    with open(scale_fold_path_b, "w", encoding="utf-8") as f:
        json.dump(scale_params_b, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_b["model"] = "M1b"
    met_fold_b["fold"] = fold_name
    met_fold_b["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_b["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1b.append(met_fold_b)
    all_test_preds_m1b.append(test_pred_fold_b)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_b)
    print("-", summary_fold_path_b)
    print("-", scale_fold_path_b)

    print("\nMétricas del fold:")
    for k, v in met_fold_b.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b
# -------------------------------------------------
metrics_m1b_df = pd.DataFrame(all_metrics_m1b)
preds_m1b_df = pd.concat(all_test_preds_m1b, ignore_index=True)

metrics_m1b_path = OUT_DIR / "HBM_O3_M1b_metrics_folds.csv"
preds_m1b_path = OUT_DIR / "HBM_O3_M1b_pred_test_all_folds.csv"

metrics_m1b_df.to_csv(metrics_m1b_path, index=False)
preds_m1b_df.to_csv(preds_m1b_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b COMPLETADA")
print("- métricas consolidadas :", metrics_m1b_path)
print("- predicciones consolidadas :", preds_m1b_path)

print("\nResumen final de métricas M1b:")
display(metrics_m1b_df)

# -------------------------------------------------
# 9) Comparar M1 vs M1b si existe archivo anterior
# -------------------------------------------------
m1_metrics_path = OUT_DIR / "HBM_O3_M1_metrics_folds.csv"

if m1_metrics_path.exists():
    metrics_m1_df = pd.read_csv(m1_metrics_path).copy()
    metrics_m1_df["model"] = "M1"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_df = pd.concat(
        [
            metrics_m1_df[cols_keep],
            metrics_m1b_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_path = OUT_DIR / "HBM_O3_compare_M1_vs_M1b.csv"
    compare_df.to_csv(compare_path, index=False)

    print("\nComparación M1 vs M1b guardada en:")
    print("-", compare_path)

    print("\nTabla comparativa:")
    display(compare_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1 previo.")
    print("Se omitió la comparación M1 vs M1b.")


Corriendo M1b - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2963 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_1_scale_params.json

Métricas del fold:
- n: 130
- mae: 2.5512620140463125
- rmse: 3.0965645851894203
- bias: 0.9180582997581269
- r: 0.776247128638095
- r2: 0.6025596047188871
- coverage_90: 0.9538461538461539
- width_90_mean: 21.841436144073185
- wis_90: 23.18656705161651
- model: M1b
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 441 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_fold_3_scale_params.json

Métricas del fold:
- n: 136
- mae: 3.8664893021797213
- rmse: 4.547608174920374
- bias: -1.908425444490483
- r: 0.6428216425231341
- r2: 0.41321966409614
- coverage_90: 0.8970588235294118
- width_90_mean: 19.253469431322927
- wis_90: 21.346395076766434
- model: M1b
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1b_pred_test_all_folds.csv

Resumen final de métricas M1b:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,130,2.551262,3.096565,0.918058,0.776247,0.602560,0.953846,21.841436,23.186567,M1b,fold_1,"2020,2021",2022
1,118,2.607638,3.212226,0.508010,0.739721,0.547188,0.983051,21.494567,21.688533,M1b,fold_2,"2020,2021,2022",2023
2,136,3.866489,4.547608,-1.908425,0.642822,0.413220,0.897059,19.253469,21.346395,M1b,fold_3,"2020,2021,2022,2023",2024



Comparación M1 vs M1b guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_compare_M1_vs_M1b.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1,fold_1,130,2.413853,2.949566,0.668166,0.787931,0.620835,0.961538,21.487811,22.818433,"2020,2021",2022
1,M1b,fold_1,130,2.551262,3.096565,0.918058,0.776247,0.602560,0.953846,21.841436,23.186567,"2020,2021",2022
2,M1,fold_2,118,2.517811,3.080924,0.178419,0.753846,0.568284,0.983051,21.137129,21.294531,"2020,2021,2022",2023
3,M1b,fold_2,118,2.607638,3.212226,0.508010,0.739721,0.547188,0.983051,21.494567,21.688533,"2020,2021,2022",2023
4,M1,fold_3,136,3.832967,4.525072,-1.852707,0.641933,0.412078,0.904412,19.998180,21.551399,"2020,2021,2022,2023",2024
5,M1b,fold_3,136,3.866489,4.547608,-1.908425,0.642822,0.413220,0.897059,19.253469,21.346395,"2020,2021,2022,2023",2024



# CELDA 6D — Comparación de diagnósticos de convergencia: M1 vs M1b

 **Objetivo:**
 comparar formalmente la calidad del muestreo bayesiano entre los modelos
 M1 y M1b usando los archivos `summary.csv` guardados por fold.

 **Qué hace esta celda:**
 1. Lee los archivos resumen de:
    - `HBM_NO2_M1_fold_*_summary.csv`
    - `HBM_NO2_M1b_fold_*_summary.csv`
 2. Extrae, para cada fold:
    - máximo `r_hat`
    - mínimo `ess_bulk`
    - mínimo `ess_tail`
 3. Consolida la comparación M1 vs M1b.
 4. Marca reglas simples de interpretación:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 - un buen modelo debe tener `r_hat` cercano a 1
 - ESS no debería ser muy bajo
 - si M1b mejora claramente estos indicadores, será preferible a M1
   aunque las métricas predictivas sean parecidas

In [9]:
# %%
# =========================
# CELDA 6D) Diagnósticos M1 vs M1b
# =========================

from pathlib import Path

summary_files = {
    "M1": {
        "fold_1": OUT_DIR / "HBM_O3_M1_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_O3_M1_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_O3_M1_fold_3_summary.csv",
    },
    "M1b": {
        "fold_1": OUT_DIR / "HBM_O3_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_O3_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_O3_M1b_fold_3_summary.csv",
    }
}

rows = []

for model_name, model_files in summary_files.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        # por seguridad, revisar que las columnas existan
        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": df_sum["r_hat"].max(),
            "min_ess_bulk": df_sum["ess_bulk"].min(),
            "min_ess_tail": df_sum["ess_tail"].min(),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows.append(row)

diag_compare = pd.DataFrame(rows).sort_values(["fold", "model"]).reset_index(drop=True)

diag_compare_path = OUT_DIR / "HBM_O3_compare_diagnostics_M1_vs_M1b.csv"
diag_compare.to_csv(diag_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_compare_diagnostics_M1_vs_M1b.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1,fold_1,1.0084,766.9505,1320.6693,True,True,True
1,M1b,fold_1,1.0017,969.2994,1407.8316,True,True,True
2,M1,fold_2,1.0105,378.4527,722.2701,False,False,True
3,M1b,fold_2,1.0047,1091.5144,1700.3808,True,True,True
4,M1,fold_3,1.0082,303.0983,682.9383,True,False,True
5,M1b,fold_3,1.0053,457.0720,1095.4011,True,True,True



# CELDA 9A — Auditoría de calidad del panel observado no2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

 **Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de no2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de no2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [10]:
# %%
# =========================
# CELDA 9A) Auditoría de calidad del panel observado O3.5
# =========================

obs_audit = pd.read_csv(OBS_CSV)

# -------------------------------------------------
# 1) Fecha mensual
# -------------------------------------------------
obs_audit["fecha"] = pd.to_datetime(
    dict(year=obs_audit["Año"].astype(int), month=obs_audit["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

# -------------------------------------------------
# 2) Filtrar O3.5 válido
# -------------------------------------------------
obs_audit["valido_O3"] = obs_audit["valido_O3"].astype(bool)

no_obs = obs_audit.loc[
    (obs_audit["valido_O3"] == True) &
    (obs_audit["O3"].notna())
].copy()

no_obs = no_obs.rename(columns={"O3": "O3_obs"})

print("Resumen general del panel observado O3.5")
print("- filas totales en archivo            :", len(obs_audit))
print("- filas válidas O3.5                :", len(no_obs))
print("- estaciones con O3.5 válido        :", no_obs["Estacion"].nunique())
print("- meses observados O3.5             :", no_obs["fecha"].nunique())
print("- rango temporal                     :", no_obs["fecha"].min().date(), "a", no_obs["fecha"].max().date())

# -------------------------------------------------
# 3) Duplicados por estación-mes
# -------------------------------------------------
dup_mask = no_obs.duplicated(subset=["Estacion", "fecha"], keep=False)
dup_rows = no_obs.loc[dup_mask].sort_values(["Estacion", "fecha"]).copy()

print("\nDuplicados por estación-mes:")
print("- número de filas duplicadas:", len(dup_rows))
print("- número de combinaciones duplicadas:",
      dup_rows[["Estacion", "fecha"]].drop_duplicates().shape[0])

# -------------------------------------------------
# 4) Resumen por estación
# -------------------------------------------------
station_summary = (
    no_obs.groupby("Estacion")["O3_obs"]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        q01=lambda s: s.quantile(0.01),
        q05=lambda s: s.quantile(0.05),
        q25=lambda s: s.quantile(0.25),
        median="median",
        q75=lambda s: s.quantile(0.75),
        q95=lambda s: s.quantile(0.95),
        q99=lambda s: s.quantile(0.99),
        max="max",
    )
    .reset_index()
    .sort_values("mean", ascending=False)
)

# -------------------------------------------------
# 5) Regla robusta IQR por estación
# -------------------------------------------------
bounds = (
    no_obs.groupby("Estacion")["O3_obs"]
    .agg(
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75)
    )
    .reset_index()
)

bounds["iqr"] = bounds["q3"] - bounds["q1"]
bounds["lower_iqr15"] = bounds["q1"] - 1.5 * bounds["iqr"]
bounds["upper_iqr15"] = bounds["q3"] + 1.5 * bounds["iqr"]
bounds["lower_iqr30"] = bounds["q1"] - 3.0 * bounds["iqr"]
bounds["upper_iqr30"] = bounds["q3"] + 3.0 * bounds["iqr"]

no_obs = no_obs.merge(bounds, on="Estacion", how="left")

no_obs["flag_iqr15"] = (
    (no_obs["O3_obs"] < no_obs["lower_iqr15"]) |
    (no_obs["O3_obs"] > no_obs["upper_iqr15"])
)

no_obs["flag_iqr30"] = (
    (no_obs["O3_obs"] < no_obs["lower_iqr30"]) |
    (no_obs["O3_obs"] > no_obs["upper_iqr30"])
)

# -------------------------------------------------
# 6) Tabla de observaciones sospechosas
# -------------------------------------------------
flagged_rows = no_obs.loc[
    no_obs["flag_iqr15"] == True,
    [
        "Estacion", "fecha", "Año", "Mes", "O3_obs",
        "Temp_media", "HR", "Vel_viento", "Presión", "Precipitación",
        "q1", "q3", "iqr", "lower_iqr15", "upper_iqr15", "flag_iqr15", "flag_iqr30"
    ]
].sort_values(["Estacion", "fecha"])

# -------------------------------------------------
# 7) Resumen de flags por estación
# -------------------------------------------------
flag_summary = (
    no_obs.groupby("Estacion")
    .agg(
        n_total=("O3_obs", "count"),
        n_flag_iqr15=("flag_iqr15", "sum"),
        n_flag_iqr30=("flag_iqr30", "sum"),
        O3_mean=("O3_obs", "mean"),
        O3_std=("O3_obs", "std"),
        O3_min=("O3_obs", "min"),
        O3_max=("O3_obs", "max")
    )
    .reset_index()
)

flag_summary["pct_flag_iqr15"] = 100 * flag_summary["n_flag_iqr15"] / flag_summary["n_total"]
flag_summary["pct_flag_iqr30"] = 100 * flag_summary["n_flag_iqr30"] / flag_summary["n_total"]

flag_summary = flag_summary.sort_values(["pct_flag_iqr15", "n_flag_iqr15"], ascending=False)

# -------------------------------------------------
# 8) Guardar auditoría
# -------------------------------------------------
station_summary_path = OUT_DIR / "HBM_O3_obs_audit_station_summary.csv"
duplicates_path = OUT_DIR / "HBM_O3_obs_audit_duplicates.csv"
flagged_rows_path = OUT_DIR / "HBM_O3_obs_audit_flagged_rows.csv"
flag_summary_path = OUT_DIR / "HBM_O3_obs_audit_flag_summary.csv"

station_summary.to_csv(station_summary_path, index=False)
dup_rows.to_csv(duplicates_path, index=False)
flagged_rows.to_csv(flagged_rows_path, index=False)
flag_summary.to_csv(flag_summary_path, index=False)

# -------------------------------------------------
# 9) Mostrar resultados
# -------------------------------------------------
print("\nResumen por estación:")
display(station_summary)

print("\nResumen de observaciones atípicas por estación:")
display(flag_summary)

print("\nPrimeras 20 observaciones sospechosas (IQR 1.5):")
display(flagged_rows.head(20))

print("\nArchivos guardados:")
print("-", station_summary_path)
print("-", duplicates_path)
print("-", flagged_rows_path)
print("-", flag_summary_path)

Resumen general del panel observado O3.5
- filas totales en archivo            : 862
- filas válidas O3.5                : 641
- estaciones con O3.5 válido        : 13
- meses observados O3.5             : 60
- rango temporal                     : 2020-01-01 a 2024-12-01

Duplicados por estación-mes:
- número de filas duplicadas: 0
- número de combinaciones duplicadas: 0

Resumen por estación:


,Estacion,n,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
12,Usaquen,51,17.937328,4.283682,11.055108,11.158857,12.153578,14.928425,17.752492,20.739756,24.240819,27.763404,29.473617
3,Fontibon,48,16.459771,4.608458,7.668927,7.794597,9.571037,13.975386,16.297133,18.557883,24.961024,28.953072,30.480650
5,Kennedy,47,15.202748,4.898863,5.461821,5.478597,6.662224,12.121556,14.886197,18.922981,21.672908,25.866897,29.263366
11,Tunal,54,14.048399,3.494724,8.308227,8.491760,8.838025,11.497216,13.896803,16.559136,19.506059,21.793331,23.465912
1,Centro de Alto Rendimiento,52,14.035078,4.393337,7.404240,7.594264,8.204113,11.580287,13.117854,15.792641,20.210945,28.971717,30.935177
7,MinAmbiente,42,13.846842,3.923727,7.355540,7.361674,7.810999,11.136097,13.475283,17.204467,20.688696,21.326248,21.688210
6,Las Ferias,58,13.173022,3.401223,7.158400,7.264181,8.843822,10.607936,12.879289,14.562837,19.715336,23.090518,25.389185
10,Suba,55,12.564030,3.845955,6.512940,6.588514,6.869007,9.847548,12.789083,14.064412,18.305968,23.614369,26.820282
4,Guaymaral,55,12.000670,3.563216,3.573851,5.310641,7.320790,9.763073,11.961096,13.746040,17.717862,22.211151,26.142073
9,San Cristobal,50,11.495471,4.033298,2.956871,3.474486,4.884314,8.914462,11.822432,14.006287,18.046626,20.812146,21.282764



Resumen de observaciones atípicas por estación:


,Estacion,n_total,n_flag_iqr15,n_flag_iqr30,O3_mean,O3_std,O3_min,O3_max,pct_flag_iqr15,pct_flag_iqr30
3,Fontibon,48,2,0,16.459771,4.608458,7.668927,30.480650,4.166667,0.000000
1,Centro de Alto Rendimiento,52,2,1,14.035078,4.393337,7.404240,30.935177,3.846154,1.923077
4,Guaymaral,55,2,1,12.000670,3.563216,3.573851,26.142073,3.636364,1.818182
10,Suba,55,2,1,12.564030,3.845955,6.512940,26.820282,3.636364,1.818182
6,Las Ferias,58,2,0,13.173022,3.401223,7.158400,25.389185,3.448276,0.000000
5,Kennedy,47,1,0,15.202748,4.898863,5.461821,29.263366,2.127660,0.000000
12,Usaquen,51,1,0,17.937328,4.283682,11.055108,29.473617,1.960784,0.000000
8,Puente Aranda,59,1,0,8.882628,2.753506,4.583796,16.402266,1.694915,0.000000
0,Bolivia,34,0,0,6.575031,2.275074,3.022345,11.045397,0.000000,0.000000
2,Colina,36,0,0,7.793799,3.017144,2.855588,13.900000,0.000000,0.000000



Primeras 20 observaciones sospechosas (IQR 1.5):


,Estacion,fecha,Año,Mes,O3_obs,Temp_media,HR,Vel_viento,Presión,Precipitación,q1,q3,iqr,lower_iqr15,upper_iqr15,flag_iqr15,flag_iqr30
19,Centro de Alto Rendimiento,2020-03-01,2020,3,30.935177,19.381613,80.223226,1.050323,NaN,106.01,11.580287,15.792641,4.212354,5.261757,22.111172,True,True
29,Centro de Alto Rendimiento,2020-04-01,2020,4,27.085255,19.681667,79.983333,0.890333,NaN,156.11,11.580287,15.792641,4.212354,5.261757,22.111172,True,False
20,Fontibon,2020-03-01,2020,3,27.230484,19.381613,80.223226,1.050323,NaN,106.01,13.975386,18.557883,4.582497,7.101640,25.431629,True,False
411,Fontibon,2023-03-01,2023,3,30.480650,18.514516,84.318710,0.870968,NaN,165.16,13.975386,18.557883,4.582497,7.101640,25.431629,True,False
21,Guaymaral,2020-03-01,2020,3,26.142073,14.388065,85.819032,0.820323,NaN,121.62,9.763073,13.746040,3.982968,3.788621,19.720492,True,True
559,Guaymaral,2024-05-01,2024,5,3.573851,14.821935,88.543871,1.094194,NaN,392.62,9.763073,13.746040,3.982968,3.788621,19.720492,True,False
22,Kennedy,2020-03-01,2020,3,29.263366,19.381613,80.223226,1.050323,NaN,106.01,12.121556,18.922981,6.801425,1.919418,29.125119,True,False
23,Las Ferias,2020-03-01,2020,3,21.356436,19.381613,80.223226,1.050323,NaN,106.01,10.607936,14.562837,3.954901,4.675585,20.495188,True,False
606,Las Ferias,2024-09-01,2024,9,25.389185,19.767667,73.347667,1.438000,NaN,85.99,10.607936,14.562837,3.954901,4.675585,20.495188,True,False
25,Puente Aranda,2020-03-01,2020,3,16.402266,19.381613,80.223226,1.050323,NaN,106.01,6.750071,10.387323,3.637252,1.294193,15.843200,True,False



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_audit_station_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_audit_duplicates.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_audit_flagged_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_audit_flag_summary.csv


# CELDA 9A — Auditoría de calidad del panel observado no2.5

 **Objetivo:**
 revisar la calidad del panel observado de estaciones antes de hacer limpieza
 o volver a ajustar el HBM.

**Qué hace esta celda:**
 1. Lee el archivo corregido de estaciones.
 2. Construye la fecha mensual.
 3. Filtra observaciones válidas de no2.5.
 4. Revisa duplicados por estación y mes.
 5. Resume la distribución de no2.5 por estación:
    - número de observaciones
    - media
    - desviación estándar
    - percentiles
    - mínimo y máximo
 6. Marca observaciones atípicas por estación usando una regla robusta IQR.
 7. Genera una tabla con las filas sospechosas.
 8. Guarda archivos de auditoría para usarlos en la siguiente etapa de limpieza.

 **Importante:**
 esta celda no elimina datos.
 Solo identifica dónde puede estar entrando ruido al modelo.

In [11]:
# %%
# =========================
# CELDA 9B) Auditoría de covariables de malla O3.5
# =========================

grid_audit = pd.read_csv(GRID_CSV)
grid_audit["fecha"] = pd.to_datetime(grid_audit["fecha"], errors="coerce")

if grid_audit["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_3km_AD_mensual_2020_2024.csv.")

covars_O3 = [
    "Vel_viento_idw",
    "diff_O3_grid",
    "grad_O3_grid",
    "adv_proxy_O3_grid",
]

missing_cols = [c for c in covars_O3 if c not in grid_audit.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en la malla: {missing_cols}")

grid_audit["year"] = grid_audit["fecha"].dt.year
grid_audit["month"] = grid_audit["fecha"].dt.month

print("Resumen general de la malla auditada")
print("- filas totales       :", len(grid_audit))
print("- celdas únicas       :", grid_audit["cell_id"].nunique())
print("- meses únicos        :", grid_audit["fecha"].nunique())
print("- rango temporal      :", grid_audit["fecha"].min().date(), "a", grid_audit["fecha"].max().date())

# -------------------------------------------------
# 1) Resumen descriptivo de covariables
# -------------------------------------------------
desc_rows = []

for col in covars_O3:
    s = pd.to_numeric(grid_audit[col], errors="coerce")
    desc_rows.append({
        "variable": col,
        "n": int(s.notna().sum()),
        "n_null": int(s.isna().sum()),
        "n_inf": int(np.isinf(s).sum()),
        "mean": float(np.nanmean(s)),
        "std": float(np.nanstd(s, ddof=1)),
        "min": float(np.nanmin(s)),
        "q01": float(np.nanquantile(s, 0.01)),
        "q05": float(np.nanquantile(s, 0.05)),
        "q25": float(np.nanquantile(s, 0.25)),
        "median": float(np.nanquantile(s, 0.50)),
        "q75": float(np.nanquantile(s, 0.75)),
        "q95": float(np.nanquantile(s, 0.95)),
        "q99": float(np.nanquantile(s, 0.99)),
        "max": float(np.nanmax(s)),
    })

desc_covars = pd.DataFrame(desc_rows)

# -------------------------------------------------
# 2) Flags IQR por covariable
# -------------------------------------------------
flag_rows = []
extreme_tables = []
monthly_flag_tables = []

for col in covars_O3:
    tmp = grid_audit[["cell_id", "fecha", "year", "month", col]].copy()
    tmp[col] = pd.to_numeric(tmp[col], errors="coerce")

    q1 = tmp[col].quantile(0.25)
    q3 = tmp[col].quantile(0.75)
    iqr = q3 - q1

    lower_15 = q1 - 1.5 * iqr
    upper_15 = q3 + 1.5 * iqr
    lower_30 = q1 - 3.0 * iqr
    upper_30 = q3 + 3.0 * iqr

    tmp["variable"] = col
    tmp["q1"] = q1
    tmp["q3"] = q3
    tmp["iqr"] = iqr
    tmp["lower_iqr15"] = lower_15
    tmp["upper_iqr15"] = upper_15
    tmp["lower_iqr30"] = lower_30
    tmp["upper_iqr30"] = upper_30

    tmp["flag_iqr15"] = (tmp[col] < lower_15) | (tmp[col] > upper_15)
    tmp["flag_iqr30"] = (tmp[col] < lower_30) | (tmp[col] > upper_30)

    n_total = tmp[col].notna().sum()
    n_flag15 = int(tmp["flag_iqr15"].sum())
    n_flag30 = int(tmp["flag_iqr30"].sum())

    flag_rows.append({
        "variable": col,
        "n_total": int(n_total),
        "n_flag_iqr15": n_flag15,
        "n_flag_iqr30": n_flag30,
        "pct_flag_iqr15": 100 * n_flag15 / n_total if n_total else np.nan,
        "pct_flag_iqr30": 100 * n_flag30 / n_total if n_total else np.nan,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower_iqr15": float(lower_15),
        "upper_iqr15": float(upper_15),
        "lower_iqr30": float(lower_30),
        "upper_iqr30": float(upper_30),
    })

    # extremos por valor absoluto
    tmp["abs_value"] = tmp[col].abs()
    top_extreme = (
        tmp.sort_values("abs_value", ascending=False)
        .head(15)
        .copy()
    )
    extreme_tables.append(top_extreme)

    # resumen mensual de flags
    monthly_flags = (
        tmp.groupby(["year", "month"])
        .agg(
            n_total=(col, "count"),
            n_flag_iqr15=("flag_iqr15", "sum"),
            n_flag_iqr30=("flag_iqr30", "sum"),
            mean_value=(col, "mean"),
            std_value=(col, "std"),
        )
        .reset_index()
    )
    monthly_flags["variable"] = col
    monthly_flags["pct_flag_iqr15"] = 100 * monthly_flags["n_flag_iqr15"] / monthly_flags["n_total"]
    monthly_flags["pct_flag_iqr30"] = 100 * monthly_flags["n_flag_iqr30"] / monthly_flags["n_total"]
    monthly_flag_tables.append(monthly_flags)

flag_summary_covars = pd.DataFrame(flag_rows).sort_values("pct_flag_iqr15", ascending=False)
extreme_covars = pd.concat(extreme_tables, ignore_index=True)
monthly_flags_covars = pd.concat(monthly_flag_tables, ignore_index=True)

# -------------------------------------------------
# 3) Correlación entre covariables
# -------------------------------------------------
corr_covars = grid_audit[covars_O3].corr(numeric_only=True)

# -------------------------------------------------
# 4) Guardar auditoría
# -------------------------------------------------
desc_covars_path = OUT_DIR / "HBM_O3_grid_audit_covariate_summary.csv"
flag_covars_path = OUT_DIR / "HBM_O3_grid_audit_flag_summary.csv"
extreme_covars_path = OUT_DIR / "HBM_O3_grid_audit_extreme_rows.csv"
monthly_flags_covars_path = OUT_DIR / "HBM_O3_grid_audit_monthly_flags.csv"
corr_covars_path = OUT_DIR / "HBM_O3_grid_audit_correlation.csv"

desc_covars.to_csv(desc_covars_path, index=False)
flag_summary_covars.to_csv(flag_covars_path, index=False)
extreme_covars.to_csv(extreme_covars_path, index=False)
monthly_flags_covars.to_csv(monthly_flags_covars_path, index=False)
corr_covars.to_csv(corr_covars_path)

# -------------------------------------------------
# 5) Mostrar resultados
# -------------------------------------------------
print("\nResumen descriptivo de covariables:")
display(desc_covars)

print("\nResumen de flags por covariable:")
display(flag_summary_covars)

print("\nCorrelación entre covariables:")
display(corr_covars)

print("\nCasos más extremos por valor absoluto:")
display(
    extreme_covars[
        ["variable", "cell_id", "fecha", "year", "month"] + covars_O3
    ].head(30)
)

print("\nMeses con mayor porcentaje de flags IQR 1.5 por covariable:")
top_months_flags = (
    monthly_flags_covars
    .sort_values(["variable", "pct_flag_iqr15"], ascending=[True, False])
    .groupby("variable")
    .head(10)
    .reset_index(drop=True)
)
display(top_months_flags)

print("\nArchivos guardados:")
print("-", desc_covars_path)
print("-", flag_covars_path)
print("-", extreme_covars_path)
print("-", monthly_flags_covars_path)
print("-", corr_covars_path)

Resumen general de la malla auditada
- filas totales       : 15240
- celdas únicas       : 254
- meses únicos        : 60
- rango temporal      : 2020-01-01 a 2024-12-01

Resumen descriptivo de covariables:


,variable,n,n_null,n_inf,mean,std,min,q01,q05,q25,median,q75,q95,q99,max
0,Vel_viento_idw,15240,0,0,1.116795e+00,0.221991,5.046784e-01,8.076667e-01,0.869237,0.944333,1.061290,1.215806,1.591290,1.717742,1.717742
1,diff_O3_grid,15240,0,0,-6.381597e-18,3.219685,-4.777906e+01,-1.098203e+01,-2.031814,-0.096547,-0.000173,0.043480,2.075238,11.248877,40.731980
2,grad_O3_grid,15240,0,0,7.861081e-05,0.000148,1.124893e-08,4.822032e-07,0.000001,0.000004,0.000015,0.000082,0.000372,0.000724,0.001661
3,adv_proxy_O3_grid,15240,0,0,8.641958e-05,0.000162,1.280564e-08,5.156865e-07,0.000001,0.000005,0.000016,0.000091,0.000422,0.000784,0.001855



Resumen de flags por covariable:


,variable,n_total,n_flag_iqr15,n_flag_iqr30,pct_flag_iqr15,pct_flag_iqr30,q1,q3,iqr,lower_iqr15,upper_iqr15,lower_iqr30,upper_iqr30
1,diff_O3_grid,15240,4964,3983,32.572178,26.135171,-0.096547,0.043480,0.140027,-0.306589,0.253521,-0.516630,0.463562
3,adv_proxy_O3_grid,15240,1866,1045,12.244094,6.856955,0.000005,0.000091,0.000086,-0.000125,0.000220,-0.000254,0.000350
2,grad_O3_grid,15240,1859,1000,12.198163,6.561680,0.000004,0.000082,0.000078,-0.000113,0.000200,-0.000230,0.000317
0,Vel_viento_idw,15240,500,0,3.280840,0.000000,0.944333,1.215806,0.271473,0.537124,1.623016,0.129914,2.030226



Correlación entre covariables:


...


Casos más extremos por valor absoluto:


,variable,cell_id,fecha,year,month,Vel_viento_idw,diff_O3_grid,grad_O3_grid,adv_proxy_O3_grid
0,Vel_viento_idw,486,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
1,Vel_viento_idw,464,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
2,Vel_viento_idw,523,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
3,Vel_viento_idw,304,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
4,Vel_viento_idw,126,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
5,Vel_viento_idw,169,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
6,Vel_viento_idw,176,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
7,Vel_viento_idw,2,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
8,Vel_viento_idw,517,2023-07-01,2023,7,1.717742,NaN,NaN,NaN
9,Vel_viento_idw,519,2023-07-01,2023,7,1.717742,NaN,NaN,NaN



Meses con mayor porcentaje de flags IQR 1.5 por covariable:


,year,month,n_total,n_flag_iqr15,n_flag_iqr30,mean_value,std_value,variable,pct_flag_iqr15,pct_flag_iqr30
0,2023,7,254,254,0,1.713691e+00,0.014138,Vel_viento_idw,100.000000,0.000000
1,2021,7,254,241,0,1.637227e+00,0.006300,Vel_viento_idw,94.881890,0.000000
2,2020,11,254,5,0,7.889624e-01,0.059452,Vel_viento_idw,1.968504,0.000000
3,2020,1,254,0,0,1.480073e+00,0.073592,Vel_viento_idw,0.000000,0.000000
4,2020,2,254,0,0,1.350505e+00,0.035660,Vel_viento_idw,0.000000,0.000000
5,2020,3,254,0,0,1.037031e+00,0.042247,Vel_viento_idw,0.000000,0.000000
6,2020,4,254,0,0,8.910268e-01,0.002204,Vel_viento_idw,0.000000,0.000000
7,2020,5,254,0,0,1.143411e+00,0.002489,Vel_viento_idw,0.000000,0.000000
8,2020,6,254,0,0,1.243268e+00,0.002327,Vel_viento_idw,0.000000,0.000000
9,2020,7,254,0,0,1.134478e+00,0.010132,Vel_viento_idw,0.000000,0.000000



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_audit_covariate_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_audit_flag_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_audit_extreme_rows.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_audit_monthly_flags.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_audit_correlation.csv



 # CELDA 9C — Construcción de covariables limpias para el HBM de no2.5

 **Objetivo:**
 crear una versión depurada y más estable de las covariables de malla
 para volver a correr el HBM con menor sensibilidad a colas extremas
 y menor redundancia entre predictores.

 **Estrategia aplicada:**
 - `Vel_viento_idw`: se conserva casi intacta
 - `diff_NO2_grid`: se transforma con signed-log y luego se winsoriza
 - `grad_NO2_grid`: se winsoriza solo para auditoría, pero no será usada
- `adv_proxy_NO2_grid`: se winsoriza y se conserva para el modelo

 **Variables finales recomendadas para el nuevo HBM:**
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Importante:**
 en esta etapa no se reentrena el HBM.
 Solo se construye y guarda la malla limpia.

In [12]:
# %%
# =========================
# CELDA 9C) Construcción de covariables limpias O3.5
# =========================

grid_clean = grid_audit.copy()

def winsorize_series(s, q_low=0.01, q_high=0.99):
    s = pd.to_numeric(s, errors="coerce").astype(float)
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    s_clip = s.clip(lower=lo, upper=hi)
    return s_clip, lo, hi

transform_report = []

# -------------------------------------------------
# 1) Velocidad del viento: conservar casi intacta
# -------------------------------------------------
grid_clean["Vel_viento_idw_clean"] = pd.to_numeric(grid_clean["Vel_viento_idw"], errors="coerce").astype(float)

transform_report.append({
    "variable_original": "Vel_viento_idw",
    "variable_limpia": "Vel_viento_idw_clean",
    "transformacion": "sin cambio",
    "q_low": np.nan,
    "q_high": np.nan
})

# -------------------------------------------------
# 2) diff_O3_grid: signed-log + winsorización
# -------------------------------------------------
x_diff = pd.to_numeric(grid_clean["diff_O3_grid"], errors="coerce").astype(float)
grid_clean["diff_O3_grid_slog"] = np.sign(x_diff) * np.log1p(np.abs(x_diff))

grid_clean["diff_O3_grid_slog_clean"], lo_diff, hi_diff = winsorize_series(
    grid_clean["diff_O3_grid_slog"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "diff_O3_grid",
    "variable_limpia": "diff_O3_grid_slog_clean",
    "transformacion": "signed_log1p + winsor_1_99",
    "q_low": float(lo_diff),
    "q_high": float(hi_diff)
})

# -------------------------------------------------
# 3) grad_O3_grid: winsorización (solo auditoría / respaldo)
# -------------------------------------------------
grid_clean["grad_O3_grid_clean"], lo_grad, hi_grad = winsorize_series(
    grid_clean["grad_O3_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "grad_O3_grid",
    "variable_limpia": "grad_O3_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_grad),
    "q_high": float(hi_grad)
})

# -------------------------------------------------
# 4) adv_proxy_O3_grid: winsorización
# -------------------------------------------------
grid_clean["adv_proxy_O3_grid_clean"], lo_adv, hi_adv = winsorize_series(
    grid_clean["adv_proxy_O3_grid"], q_low=0.01, q_high=0.99
)

transform_report.append({
    "variable_original": "adv_proxy_O3_grid",
    "variable_limpia": "adv_proxy_O3_grid_clean",
    "transformacion": "winsor_1_99",
    "q_low": float(lo_adv),
    "q_high": float(hi_adv)
})

transform_report_df = pd.DataFrame(transform_report)

# -------------------------------------------------
# 5) Definir covariables recomendadas para el nuevo HBM
# -------------------------------------------------
X_COLS_O3_CLEAN = [
    "Vel_viento_idw_clean",
    "diff_O3_grid_slog_clean",
    "adv_proxy_O3_grid_clean",
]

# -------------------------------------------------
# 6) Guardar archivo limpio
# -------------------------------------------------
grid_clean_path = OUT_DIR / "HBM_O3_grid_clean_v1.csv"
transform_report_path = OUT_DIR / "HBM_O3_grid_clean_transform_report.csv"
xcols_clean_path = OUT_DIR / "HBM_O3_grid_clean_xcols.json"

grid_clean.to_csv(grid_clean_path, index=False)
transform_report_df.to_csv(transform_report_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_O3_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 7) Resumen comparativo
# -------------------------------------------------
compare_summary = pd.DataFrame({
    "variable": [
        "Vel_viento_idw",
        "diff_O3_grid",
        "diff_O3_grid_slog_clean",
        "grad_O3_grid",
        "grad_O3_grid_clean",
        "adv_proxy_O3_grid",
        "adv_proxy_O3_grid_clean",
    ],
    "mean": [
        grid_clean["Vel_viento_idw"].mean(),
        grid_clean["diff_O3_grid"].mean(),
        grid_clean["diff_O3_grid_slog_clean"].mean(),
        grid_clean["grad_O3_grid"].mean(),
        grid_clean["grad_O3_grid_clean"].mean(),
        grid_clean["adv_proxy_O3_grid"].mean(),
        grid_clean["adv_proxy_O3_grid_clean"].mean(),
    ],
    "std": [
        grid_clean["Vel_viento_idw"].std(),
        grid_clean["diff_O3_grid"].std(),
        grid_clean["diff_O3_grid_slog_clean"].std(),
        grid_clean["grad_O3_grid"].std(),
        grid_clean["grad_O3_grid_clean"].std(),
        grid_clean["adv_proxy_O3_grid"].std(),
        grid_clean["adv_proxy_O3_grid_clean"].std(),
    ],
    "min": [
        grid_clean["Vel_viento_idw"].min(),
        grid_clean["diff_O3_grid"].min(),
        grid_clean["diff_O3_grid_slog_clean"].min(),
        grid_clean["grad_O3_grid"].min(),
        grid_clean["grad_O3_grid_clean"].min(),
        grid_clean["adv_proxy_O3_grid"].min(),
        grid_clean["adv_proxy_O3_grid_clean"].min(),
    ],
    "max": [
        grid_clean["Vel_viento_idw"].max(),
        grid_clean["diff_O3_grid"].max(),
        grid_clean["diff_O3_grid_slog_clean"].max(),
        grid_clean["grad_O3_grid"].max(),
        grid_clean["grad_O3_grid_clean"].max(),
        grid_clean["adv_proxy_O3_grid"].max(),
        grid_clean["adv_proxy_O3_grid_clean"].max(),
    ],
})

print("Covariables limpias construidas correctamente.\n")

print("Variables recomendadas para el nuevo HBM:")
print(X_COLS_O3_CLEAN)

print("\nReporte de transformaciones:")
display(transform_report_df)

print("\nResumen comparativo antes/después:")
display(compare_summary)

print("\nArchivos guardados:")
print("-", grid_clean_path)
print("-", transform_report_path)
print("-", xcols_clean_path)

Covariables limpias construidas correctamente.

Variables recomendadas para el nuevo HBM:
['Vel_viento_idw_clean', 'diff_O3_grid_slog_clean', 'adv_proxy_O3_grid_clean']

Reporte de transformaciones:


,variable_original,variable_limpia,transformacion,q_low,q_high
0,Vel_viento_idw,Vel_viento_idw_clean,sin cambio,NaN,NaN
1,diff_O3_grid,diff_O3_grid_slog_clean,signed_log1p + winsor_1_99,-2.483408e+00,2.505432
2,grad_O3_grid,grad_O3_grid_clean,winsor_1_99,4.822032e-07,0.000724
3,adv_proxy_O3_grid,adv_proxy_O3_grid_clean,winsor_1_99,5.156865e-07,0.000784



Resumen comparativo antes/después:


,variable,mean,std,min,max
0,Vel_viento_idw,1.116795e+00,0.221991,5.046784e-01,1.717742
1,diff_O3_grid,-6.381597e-18,3.219685,-4.777906e+01,40.731980
2,diff_O3_grid_slog_clean,-1.339713e-02,0.668044,-2.483408e+00,2.505432
3,grad_O3_grid,7.861081e-05,0.000148,1.124893e-08,0.001661
4,grad_O3_grid_clean,7.652834e-05,0.000136,4.822032e-07,0.000724
5,adv_proxy_O3_grid,8.641958e-05,0.000162,1.280564e-08,0.001855
6,adv_proxy_O3_grid_clean,8.424372e-05,0.000149,5.156865e-07,0.000784



Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_clean_transform_report.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_clean_xcols.json



# CELDA 9D — Reconstrucción del panel HBM con covariables limpias

 **Objetivo:**
 construir la nueva base de modelación del HBM para no2.5 usando
 las covariables limpias generadas en la celda anterior.

 **Qué hace esta celda:**
 1. Toma la malla limpia `grid_clean`.
 2. Construye `grid_base_clean` con:
    - `cell_id`
    - `fecha`
    - `year`
    - `month`
    - `cell_idx`
    - `time_idx`
    - covariables limpias recomendadas
 3. Reconstruye el panel observado válido de no2.5.
 4. Une observaciones reales con la malla limpia por:
    - `cell_id`
    - `fecha`
 5. Guarda los archivos base para volver a correr el HBM limpio.

 **Salidas principales:**
 - `HBM_NO2_grid_base_clean_v1.csv`
 - `HBM_NO2_obs_panel_clean_v1.csv`
 - `HBM_NO2_clean_xcols.json`

In [13]:
# %%
# =========================
# CELDA 9D) Reconstruir panel HBM con covariables limpias
# =========================

# -------------------------------------------------
# 1) Verificaciones mínimas
# -------------------------------------------------
required_clean_cols = [
    "cell_id", "fecha",
    "Vel_viento_idw_clean",
    "diff_O3_grid_slog_clean",
    "adv_proxy_O3_grid_clean",
]

missing_clean = [c for c in required_clean_cols if c not in grid_clean.columns]
if missing_clean:
    raise ValueError(f"Faltan columnas en grid_clean: {missing_clean}")

# asegurar fecha
grid_clean["fecha"] = pd.to_datetime(grid_clean["fecha"], errors="coerce")
if grid_clean["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en grid_clean.")

grid_clean["year"] = grid_clean["fecha"].dt.year
grid_clean["month"] = grid_clean["fecha"].dt.month

# -------------------------------------------------
# 2) Construir grid_base_clean
# -------------------------------------------------
grid_base_clean = grid_clean[
    ["cell_id", "fecha", "year", "month"] + X_COLS_O3_CLEAN
].copy()

# usar los mismos índices globales de celda y tiempo
grid_base_clean["cell_idx"] = grid_base_clean["cell_id"].map(cell_map)
grid_base_clean["time_idx"] = grid_base_clean["fecha"].map(time_map)

if grid_base_clean["cell_idx"].isna().any():
    raise ValueError("Hay cell_id en la malla limpia que no pudieron mapearse con cell_map.")
if grid_base_clean["time_idx"].isna().any():
    raise ValueError("Hay fechas en la malla limpia que no pudieron mapearse con time_map.")

grid_base_clean["cell_idx"] = grid_base_clean["cell_idx"].astype(int)
grid_base_clean["time_idx"] = grid_base_clean["time_idx"].astype(int)

# -------------------------------------------------
# 3) Reconstruir observaciones válidas O3.5
# -------------------------------------------------
obs_clean_src = pd.read_csv(OBS_CSV)

obs_clean_src["fecha"] = pd.to_datetime(
    dict(year=obs_clean_src["Año"].astype(int), month=obs_clean_src["Mes"].astype(int), day=1),
    errors="coerce"
)

if obs_clean_src["fecha"].isna().any():
    raise ValueError("Hay fechas inválidas en el panel observado.")

obs_clean_src["valido_O3"] = obs_clean_src["valido_O3"].astype(bool)

obs_no_clean = obs_clean_src.loc[
    (obs_clean_src["valido_O3"] == True) &
    (obs_clean_src["O3"].notna()) &
    (obs_clean_src["cell_id"].notna())
].copy()

obs_no_clean = obs_no_clean.rename(columns={"O3": "O3_obs"})

# -------------------------------------------------
# 4) Unir observaciones con covariables limpias
# -------------------------------------------------
obs_panel_clean = obs_no_clean.merge(
    grid_base_clean,
    on=["cell_id", "fecha"],
    how="left",
    validate="many_to_one"
)

missing_covs_clean = obs_panel_clean[X_COLS_O3_CLEAN].isna().any(axis=1).sum()
if missing_covs_clean > 0:
    raise ValueError(
        f"Hay {missing_covs_clean} observaciones sin covariables limpias. "
        "Revisa cell_id y fecha."
    )

# -------------------------------------------------
# 5) Guardar archivos base limpios
# -------------------------------------------------
grid_base_clean_path = OUT_DIR / "HBM_O3_grid_base_clean_v1.csv"
obs_panel_clean_path = OUT_DIR / "HBM_O3_obs_panel_clean_v1.csv"
xcols_clean_path = OUT_DIR / "HBM_O3_clean_xcols.json"

grid_base_clean.to_csv(grid_base_clean_path, index=False)
obs_panel_clean.to_csv(obs_panel_clean_path, index=False)

with open(xcols_clean_path, "w", encoding="utf-8") as f:
    json.dump(X_COLS_O3_CLEAN, f, ensure_ascii=False, indent=2)

# -------------------------------------------------
# 6) Resumen
# -------------------------------------------------
print("Panel HBM limpio reconstruido correctamente.\n")

print("grid_base_clean:")
print("- filas            :", len(grid_base_clean))
print("- celdas únicas    :", grid_base_clean["cell_id"].nunique())
print("- meses únicos     :", grid_base_clean["fecha"].nunique())

print("\nobs_panel_clean:")
print("- filas            :", len(obs_panel_clean))
print("- estaciones únicas:", obs_panel_clean["Estacion"].nunique())
print("- celdas observadas:", obs_panel_clean["cell_id"].nunique())

print("\nCovariables limpias usadas:")
print(X_COLS_O3_CLEAN)

print("\nArchivos guardados:")
print("-", grid_base_clean_path)
print("-", obs_panel_clean_path)
print("-", xcols_clean_path)

print("\nPrimeras filas de obs_panel_clean:")
display(
    obs_panel_clean[
        ["Estacion", "fecha", "O3_obs", "cell_id"] + X_COLS_O3_CLEAN
    ].head()
)

Panel HBM limpio reconstruido correctamente.

grid_base_clean:
- filas            : 15240
- celdas únicas    : 254
- meses únicos     : 60

obs_panel_clean:
- filas            : 641
- estaciones únicas: 13
- celdas observadas: 13

Covariables limpias usadas:
['Vel_viento_idw_clean', 'diff_O3_grid_slog_clean', 'adv_proxy_O3_grid_clean']

Archivos guardados:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_grid_base_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_obs_panel_clean_v1.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_clean_xcols.json

Primeras filas de obs_panel_clean:


,Estacion,fecha,O3_obs,cell_id,Vel_viento_idw_clean,diff_O3_grid_slog_clean,adv_proxy_O3_grid_clean
0,Centro de Alto Rendimiento,2020-01-01,17.690691,567,1.503226,-2.483408,0.000784
1,Fontibon,2020-01-01,15.080595,485,1.503226,1.642261,0.000506
2,Guaymaral,2020-01-01,13.813510,653,1.128177,0.806332,0.000053
3,Kennedy,2020-01-01,21.176695,442,1.503226,-2.223039,0.000735
4,Las Ferias,2020-01-01,11.864831,568,1.497067,2.505432,0.000743



# CELDA 9E — Validación temporal del HBM limpio (M1b-clean)

 **Objetivo:**
 volver a correr la validación temporal del HBM usando la versión limpia
 de las covariables de malla para no2.5.

 **Modelo usado:**
 - estructura: `M1b`
 - espacial: `ICAR`
 - temporal: `RW1`
 - observación: `NO2_obs`

 **Covariables limpias:**
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Qué hace esta celda:**
 1. Usa `grid_base_clean` y `obs_panel_clean`.
 2. Repite los 3 folds temporales.
 3. Escala covariables usando solo train en cada fold.
 4. Ajusta `fit_hbm_m1b_fold`.
 5. Guarda predicciones, summary y parámetros de escalamiento.
 6. Consolida métricas.
 7. Compara contra el modelo previo `M1b`.

 **Salidas principales:**
 - `HBM_NO2_M1bclean_fold_1_pred_test.csv`
 - `HBM_NO2_M1bclean_fold_2_pred_test.csv`
 - `HBM_NO2_M1bclean_fold_3_pred_test.csv`
 - `HBM_NO2_M1bclean_metrics_folds.csv`
 - `HBM_NO2_compare_M1b_vs_M1bclean.csv`

 **Importante:**
 aquí lo que más nos interesa revisar después es si bajan:
 - `width_90_mean`
 - `wis_90`
 sin deteriorar demasiado MAE/RMSE.

In [14]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_O3_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_O3_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_O3_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_O3_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_O3_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_O3_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_O3_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 1914 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 130
- mae: 2.6069995016600624
- rmse: 3.1896780614430846
- bias: 0.7741403149282209
- r: 0.7475085018552335
- r2: 0.5587689603458557
- coverage_90: 0.9461538461538461
- width_90_mean: 20.12210068196415
- wis_90: 21.721205090988406
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 2580 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_scale_params.json

Métricas del fold:
- n: 136
- mae: 3.891593579984386
- rmse: 4.603740270599134
- bias: -1.7373428646650388
- r: 0.6064783060792418
- r2: 0.3678159357447465
- coverage_90: 0.8970588235294118
- width_90_mean: 19.440418391702956
- wis_90: 21.575344367869274
- model: M1b_clean
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b-clean COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_pred_test_all_folds.csv

Resumen final de métricas M1b-clean:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,130,2.607000,3.189678,0.774140,0.747509,0.558769,0.946154,20.122101,21.721205,M1b_clean,fold_1,"2020,2021",2022
1,118,2.673405,3.341896,0.160943,0.697466,0.486458,0.983051,19.877292,20.221091,M1b_clean,fold_2,"2020,2021,2022",2023
2,136,3.891594,4.603740,-1.737343,0.606478,0.367816,0.897059,19.440418,21.575344,M1b_clean,fold_3,"2020,2021,2022,2023",2024



Comparación M1b vs M1b-clean guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_compare_M1b_vs_M1bclean.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1b,fold_1,130,2.551262,3.096565,0.918058,0.776247,0.602560,0.953846,21.841436,23.186567,"2020,2021",2022
1,M1b_clean,fold_1,130,2.607000,3.189678,0.774140,0.747509,0.558769,0.946154,20.122101,21.721205,"2020,2021",2022
2,M1b,fold_2,118,2.607638,3.212226,0.508010,0.739721,0.547188,0.983051,21.494567,21.688533,"2020,2021,2022",2023
3,M1b_clean,fold_2,118,2.673405,3.341896,0.160943,0.697466,0.486458,0.983051,19.877292,20.221091,"2020,2021,2022",2023
4,M1b,fold_3,136,3.866489,4.547608,-1.908425,0.642822,0.413220,0.897059,19.253469,21.346395,"2020,2021,2022,2023",2024
5,M1b_clean,fold_3,136,3.891594,4.603740,-1.737343,0.606478,0.367816,0.897059,19.440418,21.575344,"2020,2021,2022,2023",2024


In [15]:
# %%
# =========================
# CELDA 9E) Validación temporal HBM limpio
# =========================

all_metrics_m1bclean = []
all_test_preds_m1bclean = []

DRAWS_C = 1000
TUNE_C = 1500
CHAINS_C = 4
TARGET_ACCEPT_C = 0.99
MAX_TREEDEPTH_C = 15
RANDOM_SEED_C = 42

for fold_name, fold_info in FOLDS.items():
    print("\n" + "=" * 80)
    print(f"Corriendo M1b-clean - {fold_name}")
    print("Train years:", fold_info["train_years"])
    print("Test years :", fold_info["test_years"])

    # -------------------------------------------------
    # 1) Escalar covariables limpias usando solo train
    # -------------------------------------------------
    grid_scaled_c, scale_params_c = scale_grid_by_train_years(
        grid_df=grid_base_clean,
        x_cols=X_COLS_O3_CLEAN,
        train_years=fold_info["train_years"]
    )

    # -------------------------------------------------
    # 2) Llevar covariables escaladas al panel observado limpio
    # -------------------------------------------------
    obs_scaled_c = merge_scaled_covariates_to_obs(
        obs_df=obs_panel_clean,
        grid_scaled=grid_scaled_c,
        x_cols=X_COLS_O3_CLEAN
    )

    # -------------------------------------------------
    # 3) Ajustar modelo del fold
    # -------------------------------------------------
    idata_fold_c, test_pred_fold_c, met_fold_c = fit_hbm_m1b_fold(
        obs_scaled=obs_scaled_c,
        grid_scaled=grid_scaled_c,
        w_edges_df=w_edges,
        x_cols_raw=X_COLS_O3_CLEAN,
        train_years=fold_info["train_years"],
        test_years=fold_info["test_years"],
        n_cells=len(cell_ids),
        n_times=len(time_ids),
        draws=DRAWS_C,
        tune=TUNE_C,
        chains=CHAINS_C,
        target_accept=TARGET_ACCEPT_C,
        max_treedepth=MAX_TREEDEPTH_C,
        random_seed=RANDOM_SEED_C,
    )

    # -------------------------------------------------
    # 4) Guardar predicciones del fold
    # -------------------------------------------------
    test_pred_fold_c["fold"] = fold_name
    pred_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_pred_test.csv"
    test_pred_fold_c.to_csv(pred_fold_path_c, index=False)

    # -------------------------------------------------
    # 5) Guardar summary del posterior
    # -------------------------------------------------
    summary_fold_c = az.summary(
        idata_fold_c,
        var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
        round_to=4
    )
    summary_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_summary.csv"
    summary_fold_c.to_csv(summary_fold_path_c)

    # -------------------------------------------------
    # 6) Guardar parámetros de escalamiento
    # -------------------------------------------------
    scale_fold_path_c = OUT_DIR / f"HBM_O3_M1bclean_{fold_name}_scale_params.json"
    with open(scale_fold_path_c, "w", encoding="utf-8") as f:
        json.dump(scale_params_c, f, ensure_ascii=False, indent=2)

    # -------------------------------------------------
    # 7) Consolidar métricas
    # -------------------------------------------------
    met_fold_c["model"] = "M1b_clean"
    met_fold_c["fold"] = fold_name
    met_fold_c["train_years"] = ",".join(map(str, fold_info["train_years"]))
    met_fold_c["test_years"] = ",".join(map(str, fold_info["test_years"]))

    all_metrics_m1bclean.append(met_fold_c)
    all_test_preds_m1bclean.append(test_pred_fold_c)

    print("\nGuardado del fold:")
    print("-", pred_fold_path_c)
    print("-", summary_fold_path_c)
    print("-", scale_fold_path_c)

    print("\nMétricas del fold:")
    for k, v in met_fold_c.items():
        print(f"- {k}: {v}")

# -------------------------------------------------
# 8) Consolidar resultados M1b-clean
# -------------------------------------------------
metrics_m1bclean_df = pd.DataFrame(all_metrics_m1bclean)
preds_m1bclean_df = pd.concat(all_test_preds_m1bclean, ignore_index=True)

metrics_m1bclean_path = OUT_DIR / "HBM_O3_M1bclean_metrics_folds.csv"
preds_m1bclean_path = OUT_DIR / "HBM_O3_M1bclean_pred_test_all_folds.csv"

metrics_m1bclean_df.to_csv(metrics_m1bclean_path, index=False)
preds_m1bclean_df.to_csv(preds_m1bclean_path, index=False)

print("\n" + "=" * 80)
print("VALIDACIÓN TEMPORAL M1b-clean COMPLETADA")
print("- métricas consolidadas :", metrics_m1bclean_path)
print("- predicciones consolidadas :", preds_m1bclean_path)

print("\nResumen final de métricas M1b-clean:")
display(metrics_m1bclean_df)

# -------------------------------------------------
# 9) Comparar M1b vs M1b-clean
# -------------------------------------------------
m1b_metrics_path = OUT_DIR / "HBM_O3_M1b_metrics_folds.csv"

if m1b_metrics_path.exists():
    metrics_m1b_df = pd.read_csv(m1b_metrics_path).copy()
    metrics_m1b_df["model"] = "M1b"

    cols_keep = [
        "model", "fold", "n", "mae", "rmse", "bias", "r", "r2",
        "coverage_90", "width_90_mean", "wis_90", "train_years", "test_years"
    ]

    compare_clean_df = pd.concat(
        [
            metrics_m1b_df[cols_keep],
            metrics_m1bclean_df[cols_keep]
        ],
        ignore_index=True
    )

    compare_clean_path = OUT_DIR / "HBM_O3_compare_M1b_vs_M1bclean.csv"
    compare_clean_df.to_csv(compare_clean_path, index=False)

    print("\nComparación M1b vs M1b-clean guardada en:")
    print("-", compare_clean_path)

    print("\nTabla comparativa:")
    display(compare_clean_df.sort_values(["fold", "model"]).reset_index(drop=True))
else:
    print("\nNo se encontró el archivo de métricas de M1b previo.")
    print("Se omitió la comparación M1b vs M1b-clean.")


Corriendo M1b-clean - fold_1
Train years: [2020, 2021]
Test years : [2022]


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 1861 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_1_scale_params.json

Métricas del fold:
- n: 130
- mae: 2.6069995016600624
- rmse: 3.1896780614430846
- bias: 0.7741403149282209
- r: 0.7475085018552335
- r2: 0.5587689603458557
- coverage_90: 0.9461538461538461
- width_90_mean: 20.12210068196415
- wis_90: 21.721205090988406
- model: M1b_clean
- fold: fold_1
- train_years: 2020,2021
- test_years: 2022

Corriendo M1b-clean - fold_2
Train years: [2020, 2021, 2022]
Test years : [2023]


...

Sampling 4 chains for 1_500 tune and 1_000 draw iterations (6_000 + 4_000 draws total) took 1468 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



Guardado del fold:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_pred_test.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_summary.csv
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_fold_3_scale_params.json

Métricas del fold:
- n: 136
- mae: 3.891593579984386
- rmse: 4.603740270599134
- bias: -1.7373428646650388
- r: 0.6064783060792418
- r2: 0.3678159357447465
- coverage_90: 0.8970588235294118
- width_90_mean: 19.440418391702956
- wis_90: 21.575344367869274
- model: M1b_clean
- fold: fold_3
- train_years: 2020,2021,2022,2023
- test_years: 2024

VALIDACIÓN TEMPORAL M1b-clean COMPLETADA
- métricas consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_metrics_folds.csv
- predicciones consolidadas : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_pred_test_all_folds.csv

Resumen final de métricas M1b-clean:


,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,model,fold,train_years,test_years
0,130,2.607000,3.189678,0.774140,0.747509,0.558769,0.946154,20.122101,21.721205,M1b_clean,fold_1,"2020,2021",2022
1,118,2.673405,3.341896,0.160943,0.697466,0.486458,0.983051,19.877292,20.221091,M1b_clean,fold_2,"2020,2021,2022",2023
2,136,3.891594,4.603740,-1.737343,0.606478,0.367816,0.897059,19.440418,21.575344,M1b_clean,fold_3,"2020,2021,2022,2023",2024



Comparación M1b vs M1b-clean guardada en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_compare_M1b_vs_M1bclean.csv

Tabla comparativa:


,model,fold,n,mae,rmse,bias,r,r2,coverage_90,width_90_mean,wis_90,train_years,test_years
0,M1b,fold_1,130,2.551262,3.096565,0.918058,0.776247,0.602560,0.953846,21.841436,23.186567,"2020,2021",2022
1,M1b_clean,fold_1,130,2.607000,3.189678,0.774140,0.747509,0.558769,0.946154,20.122101,21.721205,"2020,2021",2022
2,M1b,fold_2,118,2.607638,3.212226,0.508010,0.739721,0.547188,0.983051,21.494567,21.688533,"2020,2021,2022",2023
3,M1b_clean,fold_2,118,2.673405,3.341896,0.160943,0.697466,0.486458,0.983051,19.877292,20.221091,"2020,2021,2022",2023
4,M1b,fold_3,136,3.866489,4.547608,-1.908425,0.642822,0.413220,0.897059,19.253469,21.346395,"2020,2021,2022,2023",2024
5,M1b_clean,fold_3,136,3.891594,4.603740,-1.737343,0.606478,0.367816,0.897059,19.440418,21.575344,"2020,2021,2022,2023",2024


# CELDA 9F — Comparación de diagnósticos de convergencia: M1b vs M1b-clean

 **Objetivo:**
 verificar si la limpieza de covariables no solo mejoró las métricas predictivas,
 sino también la estabilidad bayesiana del muestreo.

 **Qué hace esta celda:**
 1. Lee los archivos `summary.csv` de:
    - `M1b`
    - `M1b_clean`
 2. Extrae por fold:
    - `max_r_hat`
    - `min_ess_bulk`
    - `min_ess_tail`
 3. Consolida la comparación.
 4. Marca reglas simples:
    - `r_hat_ok`: max r_hat <= 1.01
    - `ess_bulk_ok`: min ess_bulk >= 400
    - `ess_tail_ok`: min ess_tail >= 400

 **Interpretación esperada:**
 si `M1b_clean` mantiene o mejora estos diagnósticos, será el modelo elegido
 para el ajuste final sobre todo el período.

In [16]:
# %%
# =========================
# CELDA 9F) Diagnósticos M1b vs M1b-clean
# =========================

summary_files_clean_compare = {
    "M1b": {
        "fold_1": OUT_DIR / "HBM_O3_M1b_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_O3_M1b_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_O3_M1b_fold_3_summary.csv",
    },
    "M1b_clean": {
        "fold_1": OUT_DIR / "HBM_O3_M1bclean_fold_1_summary.csv",
        "fold_2": OUT_DIR / "HBM_O3_M1bclean_fold_2_summary.csv",
        "fold_3": OUT_DIR / "HBM_O3_M1bclean_fold_3_summary.csv",
    }
}

rows_diag_clean = []

for model_name, model_files in summary_files_clean_compare.items():
    for fold_name, path_summary in model_files.items():
        if not path_summary.exists():
            print(f"No existe: {path_summary}")
            continue

        df_sum = pd.read_csv(path_summary, index_col=0)

        needed_cols = ["r_hat", "ess_bulk", "ess_tail"]
        for c in needed_cols:
            if c not in df_sum.columns:
                raise ValueError(f"Falta la columna '{c}' en {path_summary.name}")

        row = {
            "model": model_name,
            "fold": fold_name,
            "max_r_hat": float(df_sum["r_hat"].max()),
            "min_ess_bulk": float(df_sum["ess_bulk"].min()),
            "min_ess_tail": float(df_sum["ess_tail"].min()),
        }

        row["r_hat_ok"] = row["max_r_hat"] <= 1.01
        row["ess_bulk_ok"] = row["min_ess_bulk"] >= 400
        row["ess_tail_ok"] = row["min_ess_tail"] >= 400

        rows_diag_clean.append(row)

diag_clean_compare = (
    pd.DataFrame(rows_diag_clean)
    .sort_values(["fold", "model"])
    .reset_index(drop=True)
)

diag_clean_compare_path = OUT_DIR / "HBM_O3_compare_diagnostics_M1b_vs_M1bclean.csv"
diag_clean_compare.to_csv(diag_clean_compare_path, index=False)

print("Diagnósticos comparativos guardados en:")
print("-", diag_clean_compare_path)

print("\nTabla comparativa de convergencia:")
display(diag_clean_compare)

Diagnósticos comparativos guardados en:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_compare_diagnostics_M1b_vs_M1bclean.csv

Tabla comparativa de convergencia:


,model,fold,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b,fold_1,1.0017,969.2994,1407.8316,True,True,True
1,M1b_clean,fold_1,1.0047,957.5689,1712.0054,True,True,True
2,M1b,fold_2,1.0047,1091.5144,1700.3808,True,True,True
3,M1b_clean,fold_2,1.0040,815.6092,1842.6760,True,True,True
4,M1b,fold_3,1.0053,457.0720,1095.4011,True,True,True
5,M1b_clean,fold_3,1.0050,501.6013,750.2721,True,True,True


# CELDA 10 — Ajuste final del modelo seleccionado M1b-clean

 **Objetivo:**
 ajustar el modelo final seleccionado para no2.5 usando todas las
 observaciones válidas de 2020–2024 y las covariables limpias.

 **Modelo final seleccionado:**
 - observación: `NO2_obs`
 - espacial: `ICAR`
 - temporal: `RW1`
 - covariables limpias:
 - `Vel_viento_idw_clean`
 - `diff_NO2_grid_slog_clean`
 - `adv_proxy_NO2_grid_clean`

 **Qué hace esta celda:**
 1. Escala covariables limpias usando todo el período.
 2. Ajusta el modelo final `M1b_clean`.
 3. Predice sobre toda la malla 3 km y todos los meses.
 4. Exporta la superficie final:
    - `p05_hbm`
    - `p50_hbm`
    - `p95_hbm`
    - `width_90`
 5. Guarda también:
    - summary final
    - diagnósticos finales
    - parámetros de escalamiento
    - posterior en NetCDF

 **Salidas principales:**
 - `HBM_NO2_M1bclean_surface_final.csv`
 - `HBM_NO2_M1bclean_final_summary.csv`
 - `HBM_NO2_M1bclean_final_diagnostics.csv`
 - `HBM_NO2_M1bclean_final_scale_params.json`
 - `HBM_NO2_M1bclean_final_posterior.nc`

In [17]:
# %%
# =========================
# CELDA 10) Ajuste final M1b-clean
# =========================

ALL_YEARS = [2020, 2021, 2022, 2023, 2024]

DRAWS_FINAL_C = 1200
TUNE_FINAL_C = 1800
CHAINS_FINAL_C = 4
TARGET_ACCEPT_FINAL_C = 0.99
MAX_TREEDEPTH_FINAL_C = 15
RANDOM_SEED_FINAL_C = 42

# -------------------------------------------------
# 1) Escalar covariables limpias con todo el período
# -------------------------------------------------
grid_scaled_all_c, scale_params_all_c = scale_grid_by_train_years(
    grid_df=grid_base_clean,
    x_cols=X_COLS_O3_CLEAN,
    train_years=ALL_YEARS
)

obs_scaled_all_c = merge_scaled_covariates_to_obs(
    obs_df=obs_panel_clean,
    grid_scaled=grid_scaled_all_c,
    x_cols=X_COLS_O3_CLEAN
)

z_cols_c = [f"{c}_z" for c in X_COLS_O3_CLEAN]

# -------------------------------------------------
# 2) Arrays de entrenamiento completo
# -------------------------------------------------
X_all_c = obs_scaled_all_c[z_cols_c].to_numpy(dtype=float)
y_all_raw_c = obs_scaled_all_c["O3_obs"].to_numpy(dtype=float)
y_all_c = np.log(y_all_raw_c + EPS) if USE_LOG else y_all_raw_c

cell_all_c = obs_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_all_c = obs_scaled_all_c["time_idx"].to_numpy(dtype=int)

ei = w_edges["i"].to_numpy(dtype=int)
ej = w_edges["j"].to_numpy(dtype=int)

p_c = X_all_c.shape[1]
alpha_mu_all_c = float(np.mean(y_all_c))

print("Ajustando modelo final M1b-clean...")
print("- observaciones válidas:", len(obs_scaled_all_c))
print("- estaciones únicas    :", obs_scaled_all_c["Estacion"].nunique())
print("- celdas observadas     :", obs_scaled_all_c["cell_id"].nunique())
print("- celdas totales malla  :", len(cell_ids))
print("- meses totales         :", len(time_ids))
print("- covariables usadas    :", X_COLS_O3_CLEAN)

# -------------------------------------------------
# 3) Ajuste final M1b-clean
# -------------------------------------------------
with no.Model() as final_model_m1bclean:
    # efectos fijos regularizados
    alpha = no.Normal("alpha", mu=alpha_mu_all_c, sigma=1.0)
    beta = no.Normal("beta", mu=0.0, sigma=0.5, shape=p_c)

    # error observacional
    sigma_y = no.HalfNormal("sigma_y", sigma=0.75)

    # espacial ICAR
    tau_phi = no.Exponential("tau_phi", 2.0)
    phi_raw = no.Normal("phi_raw", mu=0.0, sigma=1.0, shape=len(cell_ids))
    phi = no.Deterministic("phi", phi_raw - pt.mean(phi_raw))
    no.Potential("icar_penalty", -0.5 * tau_phi * pt.sum((phi[ei] - phi[ej]) ** 2))

    # temporal RW1
    sigma_t = no.HalfNormal("sigma_t", sigma=0.25)
    delta_raw = no.GaussianRandomWalk("delta_raw", sigma=sigma_t, shape=len(time_ids))
    delta = no.Deterministic("delta", delta_raw - pt.mean(delta_raw))

    # media
    mu_all_c = alpha + pt.dot(X_all_c, beta) + phi[cell_all_c] + delta[time_all_c]

    # likelihood
    no.Normal("y_obs", mu=mu_all_c, sigma=sigma_y, observed=y_all_c)

    # muestreo
    idata_final_m1bclean = no.sample(
        draws=DRAWS_FINAL_C,
        tune=TUNE_FINAL_C,
        chains=CHAINS_FINAL_C,
        init="adapt_diag",
        target_accept=TARGET_ACCEPT_FINAL_C,
        max_treedepth=MAX_TREEDEPTH_FINAL_C,
        random_seed=RANDOM_SEED_FINAL_C,
        return_inferencedata=True,
        progressbar=True,
    )

# -------------------------------------------------
# 4) Predicción sobre toda la malla y todos los meses
# -------------------------------------------------
X_grid_all_c = grid_scaled_all_c[z_cols_c].to_numpy(dtype=float)
cell_grid_all_c = grid_scaled_all_c["cell_idx"].to_numpy(dtype=int)
time_grid_all_c = grid_scaled_all_c["time_idx"].to_numpy(dtype=int)

p05_grid_c, p50_grid_c, p95_grid_c = posterior_predict_concentration(
    idata=idata_final_m1bclean,
    X_mat=X_grid_all_c,
    cell_idx_arr=cell_grid_all_c,
    time_idx_arr=time_grid_all_c,
    eps=EPS
)

surface_final_m1bclean = grid_scaled_all_c[
    ["cell_id", "fecha", "year", "month", "cell_idx", "time_idx"]
].copy()

surface_final_m1bclean["p05_hbm"] = p05_grid_c
surface_final_m1bclean["p50_hbm"] = p50_grid_c
surface_final_m1bclean["p95_hbm"] = p95_grid_c
surface_final_m1bclean["width_90"] = (
    surface_final_m1bclean["p95_hbm"] - surface_final_m1bclean["p05_hbm"]
)

# -------------------------------------------------
# 5) Summary y diagnósticos finales
# -------------------------------------------------
summary_final_m1bclean = az.summary(
    idata_final_m1bclean,
    var_names=["alpha", "beta", "sigma_y", "tau_phi", "sigma_t"],
    round_to=4
)

diag_final_m1bclean = pd.DataFrame([{
    "model": "M1b_clean_final",
    "max_r_hat": float(summary_final_m1bclean["r_hat"].max()),
    "min_ess_bulk": float(summary_final_m1bclean["ess_bulk"].min()),
    "min_ess_tail": float(summary_final_m1bclean["ess_tail"].min()),
    "r_hat_ok": bool(summary_final_m1bclean["r_hat"].max() <= 1.01),
    "ess_bulk_ok": bool(summary_final_m1bclean["ess_bulk"].min() >= 400),
    "ess_tail_ok": bool(summary_final_m1bclean["ess_tail"].min() >= 400),
}])

# -------------------------------------------------
# 6) Guardar salidas
# -------------------------------------------------
surface_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_surface_final.csv"
summary_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_final_summary.csv"
diag_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_final_diagnostics.csv"
scale_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_final_scale_params.json"
idata_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_final_posterior.nc"

surface_final_m1bclean.to_csv(surface_final_clean_path, index=False)
summary_final_m1bclean.to_csv(summary_final_clean_path)
diag_final_m1bclean.to_csv(diag_final_clean_path, index=False)

with open(scale_final_clean_path, "w", encoding="utf-8") as f:
    json.dump(scale_params_all_c, f, ensure_ascii=False, indent=2)

az.to_netcdf(idata_final_m1bclean, idata_final_clean_path)

# -------------------------------------------------
# 7) Resumen final
# -------------------------------------------------
print("\nAJUSTE FINAL M1b-clean COMPLETADO")
print("- superficie final     :", surface_final_clean_path)
print("- summary final        :", summary_final_clean_path)
print("- diagnósticos finales :", diag_final_clean_path)
print("- scale params         :", scale_final_clean_path)
print("- posterior netcdf     :", idata_final_clean_path)

print("\nDiagnósticos finales:")
display(diag_final_m1bclean)

print("\nPrimeras filas de la superficie final limpia:")
display(surface_final_m1bclean.head())

Ajustando modelo final M1b-clean...
- observaciones válidas: 641
- estaciones únicas    : 13
- celdas observadas     : 13
- celdas totales malla  : 254
- meses totales         : 60
- covariables usadas    : ['Vel_viento_idw_clean', 'diff_O3_grid_slog_clean', 'adv_proxy_O3_grid_clean']


Initializing NUTS using adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: local_subtensor_merge
ERROR (pytensor.graph.rewriting.basic): node: Subtensor{i}(Subtensor{start:}.0, 0)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 1920, in process_node
    replacements = node_rewriter.transform(
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\graph\rewriting\basic.py", line 993, in transform
    return self.fn(fgraph, node)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 416, in local_subtensor_merge
    merge_two_slices(
  File "d:\TRABAJO DE GRADO BEN-MAP\.venv\Lib\site-packages\pytensor\tensor\rewriting\subtensor.py", line 865, in

Output()

Sampling 4 chains for 1_800 tune and 1_200 draw iterations (7_200 + 4_800 draws total) took 4436 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



AJUSTE FINAL M1b-clean COMPLETADO
- superficie final     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_surface_final.csv
- summary final        : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_final_summary.csv
- diagnósticos finales : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_final_diagnostics.csv
- scale params         : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_final_scale_params.json
- posterior netcdf     : d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_final_posterior.nc

Diagnósticos finales:


,model,max_r_hat,min_ess_bulk,min_ess_tail,r_hat_ok,ess_bulk_ok,ess_tail_ok
0,M1b_clean_final,1.0068,520.7988,987.8027,True,True,True



Primeras filas de la superficie final limpia:


,cell_id,fecha,year,month,cell_idx,time_idx,p05_hbm,p50_hbm,p95_hbm,width_90
0,0,2020-01-01,2020,1,0,0,2.503180,13.779391,72.153779,69.650599
1,1,2020-01-01,2020,1,1,0,2.566153,13.661728,78.929238,76.363085
2,2,2020-01-01,2020,1,2,0,2.683338,14.130858,74.703478,72.020139
3,41,2020-01-01,2020,1,3,0,2.338490,13.447773,75.608877,73.270386
4,42,2020-01-01,2020,1,4,0,2.517089,13.773598,77.750612,75.233523



# CELDA 10B — Diagnóstico focal del summary final del modelo limpio

 **Objetivo:**
 identificar qué parámetro(s) del modelo final `M1b_clean` están
 generando el `max_r_hat` más alto y revisar si el problema es puntual
 o extendido.

 **Qué hace esta celda:**
 1. Lee `HBM_NO2_M1bclean_final_summary.csv`.
 2. Ordena el summary por `r_hat` de mayor a menor.
 3. Marca parámetros con:
    - `r_hat > 1.01`
    - `ess_bulk < 400`
    - `ess_tail < 400`
 4. Muestra:
    - top 20 parámetros con peor `r_hat`
    - subconjunto de parámetros problemáticos
 5. Guarda una tabla de diagnóstico focal.

 **Interpretación esperada:**
 - si solo 1 o pocos parámetros quedan apenas por encima de 1.01,
   el modelo sigue siendo razonablemente usable;
 - si son muchos, habría que volver a ajustar.

In [18]:
# %%
# =========================
# CELDA 10B) Diagnóstico focal del summary final limpio
# =========================

summary_final_clean_path = OUT_DIR / "HBM_O3_M1bclean_final_summary.csv"
summary_final_clean = pd.read_csv(summary_final_clean_path, index_col=0)

needed_cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
missing_cols = [c for c in needed_cols if c not in summary_final_clean.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas en el summary final: {missing_cols}")

summary_diag = summary_final_clean.copy().reset_index().rename(columns={"index": "parametro"})

summary_diag["flag_rhat"] = summary_diag["r_hat"] > 1.01
summary_diag["flag_ess_bulk"] = summary_diag["ess_bulk"] < 400
summary_diag["flag_ess_tail"] = summary_diag["ess_tail"] < 400

summary_diag["n_flags"] = (
    summary_diag["flag_rhat"].astype(int) +
    summary_diag["flag_ess_bulk"].astype(int) +
    summary_diag["flag_ess_tail"].astype(int)
)

# top por rhat
top_rhat = summary_diag.sort_values(["r_hat", "ess_bulk"], ascending=[False, True]).head(20).copy()

# parámetros problemáticos
problem_params = summary_diag.loc[
    (summary_diag["flag_rhat"]) |
    (summary_diag["flag_ess_bulk"]) |
    (summary_diag["flag_ess_tail"])
].sort_values(["n_flags", "r_hat", "ess_bulk"], ascending=[False, False, True]).copy()

# guardar
summary_diag_path = OUT_DIR / "HBM_O3_M1bclean_final_summary_diagnostic_focus.csv"
summary_diag.to_csv(summary_diag_path, index=False)

print("Archivo guardado:")
print("-", summary_diag_path)

print("\nResumen global del summary final:")
print("- número total de parámetros        :", len(summary_diag))
print("- parámetros con r_hat > 1.01       :", int(summary_diag["flag_rhat"].sum()))
print("- parámetros con ess_bulk < 400     :", int(summary_diag["flag_ess_bulk"].sum()))
print("- parámetros con ess_tail < 400     :", int(summary_diag["flag_ess_tail"].sum()))

print("\nTop 20 parámetros con mayor r_hat:")
display(top_rhat[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "n_flags"]])

print("\nParámetros problemáticos:")
display(problem_params[["parametro", "mean", "sd", "ess_bulk", "ess_tail", "r_hat", "flag_rhat", "flag_ess_bulk", "flag_ess_tail"]])

Archivo guardado:
- d:\TRABAJO DE GRADO BEN-MAP\CODIGO\HBM_O3_V2_OUT\HBM_O3_M1bclean_final_summary_diagnostic_focus.csv

Resumen global del summary final:
- número total de parámetros        : 7
- parámetros con r_hat > 1.01       : 0
- parámetros con ess_bulk < 400     : 0
- parámetros con ess_tail < 400     : 0

Top 20 parámetros con mayor r_hat:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,n_flags
0,alpha,2.4478,0.2606,520.7988,987.8027,1.0068,0
6,sigma_t,0.2396,0.0238,8437.6596,3453.7064,1.0025,0
1,beta[0],0.0179,0.0284,11682.6529,3451.3604,1.0018,0
5,tau_phi,0.0013,0.0013,5800.3815,2264.4564,1.0007,0
3,beta[2],0.0006,0.0063,12129.9811,3455.5213,1.0004,0
4,sigma_y,0.1593,0.0047,7310.3761,3476.6614,1.0003,0
2,beta[1],-0.0534,0.0032,11123.2888,3057.7442,1.0000,0



Parámetros problemáticos:


,parametro,mean,sd,ess_bulk,ess_tail,r_hat,flag_rhat,flag_ess_bulk,flag_ess_tail
